# SkyAI Virtual Jewellery Try-On — Rendering Pipeline
### Colab Pro notebook · built incrementally in 3 steps, per your instructions

**This section: Step 1 — Environment Setup + Segmentation (Stage 1)**

Stages 2–4 (anchor detection, deterministic placement, and bounded generative harmonisation + LoRA training) are added in later steps, once you've reviewed this one and said "next."

## Before you run this — what I flagged while reading the brief

You asked me to push back rather than silently build around problems. Here's everything that needed a judgment call, in the order you'll hit it below.

**1. I found a materially better way to load SAM2 than what the brief implies.**
The brief's phrasing ("load the largest SAM2 checkpoint...") reads like the classic workflow: clone `facebookresearch/sam2`, `pip install -e .`, download raw `.pt` checkpoints, resolve Hydra config YAMLs by hand. I checked, and `transformers` now has **native SAM2.1 support** (`Sam2Model` / `Sam2Processor`), with Meta's official checkpoints hosted at `facebook/sam2.1-hiera-{tiny,small,base-plus,large}` and loadable with a single `from_pretrained(repo_id)` call — no repo clone, no manual checkpoint plumbing, no Hydra. I used that path instead. It's simpler, less brittle, and keeps everything (SAM2, Grounding DINO, ViTMatte) on one library. If you specifically want the standalone `sam2` package instead (e.g. for its video-tracking features, which we don't need here), say so and I'll swap it.

**2. `segment_jewellery(image, prompt)` — what is `prompt`, exactly?**
SAM2 itself isn't promptable by text at all — only points, boxes, or masks. So "prompt" in the brief is ambiguous between (a) a point/box you already have, e.g. from a UI click, or (b) a text label like `"a diamond ring"`, which is what an unattended catalogue-ingestion pipeline would actually need. I implemented both: a string runs through **Grounding DINO** (open-vocabulary text→box detection) to get a box, then SAM2 segments that box; a `{"box": ...}` or `{"point": ...}` dict skips Grounding DINO and goes straight to SAM2. This is a real design decision, not just plumbing — if your product pipeline will always have a human click/crop step, the text path (and the Grounding DINO dependency it brings) may be unnecessary weight. Worth confirming before Step 2 builds on top of this.

**3. "Load the largest checkpoint that fits VRAM" is nearly a non-issue today, but I built it anyway.**
All four SAM2.1 sizes are small — tiny 38.9M, small 46M, base_plus 80.8M, large 224.4M params — so even "large" is under 1GB. On any real Colab GPU (T4/L4/A100) the VRAM-gated selector below will *always* pick `large`. I implemented it as specified regardless, both as a sane fallback (e.g. a constrained/shared GPU) and because it's the exact pattern Stage 4 will need for real, where model size vs. VRAM is a genuine constraint.

**4. ViTMatte needs a trimap — the brief never says how to make one.**
ViTMatte doesn't take an image alone; it takes an image + a 3-region trimap (definite foreground / definite background / unknown), and only refines the unknown band. Nothing upstream in the brief produces this. I added `mask_to_trimap()`, which bootstraps one from the SAM2 binary mask by eroding it for "definite foreground" and dilating it for the "unknown" band — the standard way to do this without manual trimap annotation. Without this step, the ViTMatte call the brief describes simply has no valid input.

**5. GPU tier: this doesn't bite in Step 1, but it will in Step 3 — flagging now as you asked.**
FLUX.1 Fill [dev] (Stage 4) is a ~12B-parameter model — roughly 24GB at bf16 for weights alone, which lines up with the brief's own LoRA-training note ("single 24GB-class GPU"). Colab Pro can hand you a T4 (16GB), L4 (24GB), or A100 (40GB), and those aren't equivalent for what's coming:
- **T4** — Stage 4 inference needs aggressive offloading and will be slow; LoRA training as specified likely won't fit without cutting resolution/batch size hard.
- **L4** — right at the brief's stated bar; workable with gradient checkpointing and an 8-bit optimizer for margin.
- **A100** — comfortable for both, no exotic tricks needed.

The GPU-detection cell below prints a tailored version of this and saves it to a small state file in Drive so Steps 2–3 can read it back without re-diagnosing.

**6. Smaller calls I made without asking, in case you want to override them:** `hustvl/vitmatte-small-composition-1k` over `-base-` (faster, minor accuracy gap for clean product-shot edges — one-line swap either way); `IDEA-Research/grounding-dino-base` over `-tiny` (more accurate, still small); a real public-domain Wikimedia Commons photo as the default smoke-test image rather than a synthetic drawing, since a flat vector shape doesn't meaningfully exercise either Grounding DINO or SAM2 (both trained on real photography) — with a synthetic fallback if the download fails, and a Drive-path override if you drop in a real catalogue photo.

**7. HF Hub cache redirected to Drive.** Instruction 1 asked for a persistent Drive directory for checkpoints/datasets because Colab's local disk doesn't survive a session reset. Rather than manually managing download paths per model, I pointed `HF_HOME` at a folder in that Drive directory before any model import — so SAM2 / Grounding DINO / ViTMatte now, and FLUX.1 Fill / Qwen-Image-Edit in Step 3 (multi-GB each), only ever download once, full stop, regardless of session resets.

### 1 · Colab environment setup

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    # Lets this notebook be sanity-checked outside Colab (e.g. locally) without crashing.
    print("Not running in Colab -- skipping Drive mount, using ./drive_mirror instead.")
    DRIVE_ROOT = "./drive_mirror"

PROJECT_DIR    = os.path.join(DRIVE_ROOT, "SkyAI_TryOn_Pipeline")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")   # LoRA / fine-tune checkpoints (Step 3)
DATASET_DIR    = os.path.join(PROJECT_DIR, "datasets")      # training pairs, sample catalogue images
OUTPUT_DIR     = os.path.join(PROJECT_DIR, "outputs")       # rendered results, visual test outputs
HF_CACHE_DIR   = os.path.join(PROJECT_DIR, "hf_cache")      # redirected HF Hub cache (see note below)
STATE_PATH     = os.path.join(PROJECT_DIR, "pipeline_state.json")

for d in (PROJECT_DIR, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR, HF_CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# Colab's local disk (/root/.cache/huggingface by default) does NOT survive a
# session reset, so every downloaded model gets re-fetched on every fresh
# runtime. Redirecting HF_HOME into the persistent Drive folder means SAM2 /
# Grounding DINO / ViTMatte now -- and FLUX.1 Fill / Qwen-Image-Edit in
# Step 3, which are multi-GB -- only download once, ever. Must be set BEFORE
# any `transformers`/`huggingface_hub` import below.
os.environ["HF_HOME"] = HF_CACHE_DIR

print(f"Project directory   : {PROJECT_DIR}")
print(f"HF Hub cache (Drive) : {HF_CACHE_DIR}")

In [ ]:
import json

# Small persisted-state helper. Local Python variables (and Colab's local
# disk) don't survive a session reset. This keeps a tiny JSON of pipeline
# decisions (detected GPU, chosen SAM2 checkpoint, etc.) in Drive so later
# steps -- and any future run after a disconnect -- don't have to re-derive
# them from scratch, and so Step 3's training section can resume cleanly.

def load_state() -> dict:
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "r") as f:
            return json.load(f)
    return {}

def save_state(**updates) -> dict:
    state = load_state()
    state.update(updates)
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)
    return state

state = load_state()
if state:
    print("Loaded existing pipeline state:")
    print(json.dumps(state, indent=2))
else:
    print("No existing state file yet -- starting fresh.")

In [ ]:
import torch


def detect_gpu() -> dict:
    if not torch.cuda.is_available():
        return {"has_gpu": False, "name": None, "vram_gb": 0.0}
    name = torch.cuda.get_device_name(0)
    total_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = round(total_bytes / (1024 ** 3), 1)
    return {"has_gpu": True, "name": name, "vram_gb": vram_gb}


gpu_info = detect_gpu()
save_state(gpu=gpu_info, torch_version=torch.__version__)

if not gpu_info["has_gpu"]:
    print(
        "No GPU detected. Go to Runtime > Change runtime type > select a GPU "
        "(T4 / L4 / A100), then re-run this cell. Stage 1 will technically run "
        "on CPU but very slowly, and Stage 4 (Step 3) needs a GPU outright."
    )
else:
    print(f"GPU   : {gpu_info['name']}")
    print(f"VRAM  : {gpu_info['vram_gb']} GB")
    print(f"torch : {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

    vram = gpu_info["vram_gb"]
    print("\n--- What this GPU tier means for the rest of the pipeline ---")
    print(
        "Stage 1 (SAM2) + Stage 2 (MediaPipe) + Stage 3 (OpenCV warp): comfortable "
        "on any of T4 / L4 / A100 -- all three stages are lightweight."
    )
    if vram < 20:  # T4-class (~16GB)
        print(
            "Stage 4 (FLUX.1 Fill [dev], ~12B params) will need aggressive offloading "
            "(CPU/sequential offload or 4-bit weights) and will be noticeably slow per "
            "render on this tier.\n"
            "LoRA training as the brief specifies it (targeting a '24GB-class GPU') will "
            "likely NOT fit as-is here -- expect to cut training resolution/batch size "
            "hard, or run training on a bigger GPU and use this one only for inference."
        )
    elif vram < 32:  # L4-class (~24GB)
        print(
            "Stage 4 inference should work with bf16 weights plus light offloading of the "
            "text encoders.\n"
            "LoRA training sits right at the brief's stated bar (24GB-class) -- plan on "
            "gradient checkpointing and an 8-bit optimizer for safety margin, since nominal "
            "VRAM is rarely all usable once PyTorch's allocator overhead is accounted for."
        )
    else:  # A100-class (~40GB+)
        print(
            "Comfortable headroom for both Stage 4 inference and LoRA training as "
            "specified -- standard bf16 + gradient checkpointing should be enough, no "
            "exotic memory tricks required."
        )

### 2 · Dependencies (Step 1 only — Stage 1 segmentation)

In [ ]:
# --- Step 1 dependencies (Stage 1: segmentation) ---
# Deliberately NOT touching torch / torchvision here: Colab's preinstalled
# build is already matched to the assigned GPU's CUDA driver, and
# force-reinstalling it is a common way to quietly break the runtime. We only
# checked its version in the cell above.
#
# Pins below are known-good as of early 2026. If a future Colab base image
# makes one of these fail to resolve, relax that single pin (drop the exact
# version, keep the lower bound) rather than the whole line.

!pip install -q -U \
    "transformers>=4.56" \
    "accelerate>=0.34" \
    "huggingface_hub>=0.26" \
    "opencv-python-headless>=4.10" \
    "pillow>=10.4" \
    "matplotlib>=3.9" \
    "timm>=1.0.9"

print("Install complete.")

### 3 · Load Stage 1 models

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch

try:
    from transformers import Sam2Model, Sam2Processor
except ImportError as e:
    raise ImportError(
        "This transformers version doesn't expose Sam2Model/Sam2Processor yet. "
        "Run: pip install -U transformers   then Runtime > Restart session, and "
        "re-run from the installs cell."
    ) from e

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# SAM2.1 checkpoints, official facebook/* repos on the HF Hub -- loaded
# directly via from_pretrained, no manual checkpoint download or Hydra config
# resolution needed (see the flagged note above on why this replaces the
# git-clone-the-Meta-repo approach).
SAM2_VARIANTS = {
    # size name -> (HF repo id, approx params (M), min VRAM (GB) to "unlock")
    "tiny":      ("facebook/sam2.1-hiera-tiny",      38.9,   0),
    "small":     ("facebook/sam2.1-hiera-small",      46.0,   4),
    "base_plus": ("facebook/sam2.1-hiera-base-plus",  80.8,   8),
    "large":     ("facebook/sam2.1-hiera-large",     224.4,  12),
}


def select_sam2_variant(vram_gb: float) -> str:
    """Largest SAM2.1 variant whose min-VRAM bar the detected GPU clears.

    Note: even 'large' is under ~1GB in fp32, so on any real Colab GPU
    (T4/L4/A100) this always resolves to 'large'. It's kept VRAM-gated
    per spec anyway -- both as a safe fallback (e.g. a constrained/shared
    GPU) and because it's the exact pattern Stage 4 will actually need.
    """
    choice = "tiny"
    for name, (_, _, min_vram) in SAM2_VARIANTS.items():
        if vram_gb >= min_vram:
            choice = name
    return choice


sam2_variant = select_sam2_variant(gpu_info["vram_gb"]) if gpu_info["has_gpu"] else "tiny"
sam2_repo_id, sam2_params_m, _ = SAM2_VARIANTS[sam2_variant]
print(f"Selected SAM2.1 variant: {sam2_variant} (~{sam2_params_m}M params) from {sam2_repo_id}")

# Loaded at fp32 for correctness-first behaviour: the brief's non-negotiable
# constraint is pixel-accurate geometry, so Step 1 prioritises a mask that's
# unambiguously right over a faster bf16 pass. Once Step 2's fidelity metrics
# confirm the pipeline works, switching sam2_model to bf16 (and casting the
# processor's pixel_values to match) is a straightforward speed win.
sam2_model = Sam2Model.from_pretrained(sam2_repo_id).to(DEVICE)
sam2_model.eval()
sam2_processor = Sam2Processor.from_pretrained(sam2_repo_id)

save_state(sam2_variant=sam2_variant, sam2_repo_id=sam2_repo_id, device=DEVICE)
print(f"SAM2 loaded on {DEVICE}.")

## Where jewellery-specific SAM2 fine-tuning plugs in later

We don't have the annotated dataset yet (the brief calls for a few hundred–2,000 pieces across categories), so there's nothing to train on today — but here's exactly where it slots in once that dataset exists, without touching anything else in this pipeline:

- **What changes:** only `sam2_model` gets replaced with a fine-tuned checkpoint. `sam2_processor`, `segment_jewellery()`, and everything downstream (Grounding DINO, ViTMatte, and all of Stages 2–4) stays identical — this is exactly why each stage was built as an independent, swappable function per your general rules.
- **What to fine-tune:** typically just the mask decoder (small, fast to train, and where jewellery-specific failure modes like clipped chain links and bled prong settings actually live) rather than the full image encoder. A LoRA on top of the frozen encoder + a fully fine-tuned decoder is a common middle ground if the decoder-only fit underperforms.
- **Data format:** image + point/box prompt + ground-truth binary mask triplets, in the same structure Meta's own SAM2 training scripts expect (their repo includes a `training/` module for exactly this).
- **Where the swap happens in this notebook:** replace the `sam2_repo_id` passed to `Sam2Model.from_pretrained(...)` in the SAM2-loading cell with a path to your fine-tuned checkpoint (local, or pushed to a private HF Hub repo) — one line, nothing else in the pipeline needs to change.
- **How you'd know it's needed:** watch the Stage 2 fidelity metrics (SSIM/LPIPS, ΔE) once real catalogue photos are running through this — if failures cluster on thin chains or faceted stones specifically (rather than being spread evenly), that's the signal this fine-tuning step has become worth the annotation cost.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# Resolves a text prompt ("a diamond ring") to a bounding box, which is then
# handed to SAM2 -- see the flagged note above on why this exists at all
# (SAM2 itself only takes point/box/mask prompts, not text).
#
# "-base" for accuracy; swap to "IDEA-Research/grounding-dino-tiny" for a
# lighter/faster model if you're doing high-volume catalogue ingestion and
# can tolerate slightly less precise boxes (SAM2 + the box-margin handling
# below is fairly forgiving of an imperfect box).
DINO_MODEL_ID = "IDEA-Research/grounding-dino-base"

dino_processor = AutoProcessor.from_pretrained(DINO_MODEL_ID)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_ID).to(DEVICE)
dino_model.eval()

save_state(dino_model_id=DINO_MODEL_ID)
print(f"Grounding DINO ({DINO_MODEL_ID}) loaded on {DEVICE}.")

In [ ]:
from transformers import VitMatteImageProcessor, VitMatteForImageMatting

# "small" is noticeably faster than "base" with only a minor accuracy gap for
# this use case (cleaning hairline chain/prong edges on a product shot, not
# complex natural-image matting) -- see the flagged note above. Swap to
# "hustvl/vitmatte-base-composition-1k" for a one-line A/B if you want to
# check whether the gap matters on real catalogue photos.
VITMATTE_MODEL_ID = "hustvl/vitmatte-small-composition-1k"

vitmatte_processor = VitMatteImageProcessor.from_pretrained(VITMATTE_MODEL_ID)
vitmatte_model = VitMatteForImageMatting.from_pretrained(VITMATTE_MODEL_ID).to(DEVICE)
vitmatte_model.eval()

save_state(vitmatte_model_id=VITMATTE_MODEL_ID)
print(f"ViTMatte ({VITMATTE_MODEL_ID}) loaded on {DEVICE}.")

### 4 · Segmentation pipeline functions
`segment_jewellery(image, prompt) -> dict` — the Stage 1 deliverable the brief asks for — is assembled from the four independent pieces below, each swappable on its own per your general rules.

In [ ]:
def ground_text_to_box(
    image: Image.Image,
    text_prompt: str,
    box_threshold: float = 0.35,
    text_threshold: float = 0.25,
) -> tuple:
    """Resolve an open-vocabulary text prompt to a single best bounding box.

    Grounding DINO requires the query to be lowercase and end with a period
    -- easy to miss and it silently returns nothing if you don't do this.
    Returns (box, score) where box = [x0, y0, x1, y1] in pixel coordinates.
    Raises ValueError if nothing was detected above threshold.
    """
    text = text_prompt.lower().strip()
    if not text.endswith("."):
        text += "."

    inputs = dino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = dino_model(**inputs)

    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]],  # (height, width)
    )[0]

    if len(results["boxes"]) == 0:
        raise ValueError(
            f"Grounding DINO found nothing for prompt '{text_prompt}' "
            f"(box_threshold={box_threshold}). Try a looser threshold, a "
            f"different phrase, or pass an explicit box/point instead."
        )

    best_idx = int(results["scores"].argmax())
    box = [round(v, 2) for v in results["boxes"][best_idx].tolist()]
    score = float(results["scores"][best_idx])
    return box, score

In [ ]:
def sam2_binary_mask(image: Image.Image, box=None, point=None, point_label: int = 1) -> np.ndarray:
    """Run SAM2 with either a box prompt or a point prompt.

    Exactly one of box / point should be given.
      box:   [x0, y0, x1, y1] in pixel coordinates.
      point: [x, y] in pixel coordinates.

    SAM2 returns 3 candidate masks per prompt ranked by predicted quality;
    this picks the highest-IoU one, which is the standard way to use it for
    a single object (see the model's own multi-point-refinement example).

    Returns a boolean HxW numpy array.
    """
    if (box is None) == (point is None):
        raise ValueError("sam2_binary_mask needs exactly one of box= or point=.")

    if box is not None:
        inputs = sam2_processor(images=image, input_boxes=[[box]], return_tensors="pt")
    else:
        inputs = sam2_processor(
            images=image,
            input_points=[[[point]]],
            input_labels=[[[point_label]]],
            return_tensors="pt",
        )
    inputs = inputs.to(DEVICE)

    with torch.no_grad():
        outputs = sam2_model(**inputs, multimask_output=True)

    masks = sam2_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs["original_sizes"]
    )[0]  # shape: (num_objects, num_candidate_masks, H, W)

    iou = outputs.iou_scores.cpu()[0, 0]  # (num_candidate_masks,) for object 0
    best = int(iou.argmax())
    return masks[0, best].numpy().astype(bool)

In [ ]:
def mask_to_trimap(binary_mask: np.ndarray, erode_px: int = 15, dilate_px: int = 15) -> Image.Image:
    """Bootstrap a 3-region trimap (fg / bg / unknown) from a coarse binary mask.

    ViTMatte requires an image + trimap pair, not just an image -- nothing in
    the brief's Stage 1 description produces a trimap, so this fills that gap
    (see the flagged note above). Definite-foreground = eroded mask,
    definite-background = outside the dilated mask, everything in between
    ('unknown') is the band ViTMatte actually refines -- which is exactly the
    hairline chain-link / gem-facet boundary we care about.
    """
    mask_u8 = binary_mask.astype(np.uint8) * 255
    kernel_e = np.ones((erode_px, erode_px), np.uint8)
    kernel_d = np.ones((dilate_px, dilate_px), np.uint8)

    sure_fg = cv2.erode(mask_u8, kernel_e, iterations=1)
    sure_bg = 255 - cv2.dilate(mask_u8, kernel_d, iterations=1)

    trimap = np.full(mask_u8.shape, 128, dtype=np.uint8)  # default: unknown
    trimap[sure_fg > 0] = 255
    trimap[sure_bg > 0] = 0
    return Image.fromarray(trimap, mode="L")


def refine_with_vitmatte(image: Image.Image, binary_mask: np.ndarray) -> tuple:
    """Run the trimap through ViTMatte to get a soft alpha matte.

    Returns (alpha, trimap) -- alpha as a float32 HxW array in [0, 1],
    trimap as the PIL image that produced it (handy for the visual check).
    """
    trimap = mask_to_trimap(binary_mask)

    inputs = vitmatte_processor(images=image, trimaps=trimap, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        alphas = vitmatte_model(**inputs).alphas

    alpha = alphas[0, 0].cpu().numpy()
    # VitMatteImageProcessor pads to a stride multiple internally; crop back
    # down to the trimap's actual size before returning.
    w, h = trimap.size
    alpha = alpha[:h, :w]
    return np.clip(alpha, 0.0, 1.0).astype(np.float32), trimap

In [ ]:
def segment_jewellery(image: Image.Image, prompt, refine: bool = True, verbose: bool = False) -> dict:
    """Stage 1: extract a clean, pixel-accurate alpha mask of a jewellery piece.

    `prompt` accepts three forms (see the flagged note above on why):
      - str                          -> open-vocabulary text, e.g. "a diamond ring".
                                         Resolved to a box via Grounding DINO.
      - {"box": [x0, y0, x1, y1]}    -> used directly, skips Grounding DINO.
      - {"point": [x, y]}            -> used directly, skips Grounding DINO.
                                         Optional "label": 1 (fg, default) or 0 (bg).

    Returns a dict with:
      alpha_mask   : float32 HxW array in [0, 1] -- the final output.
      binary_mask  : bool HxW array -- the raw SAM2 mask, pre-matting.
      trimap       : PIL Image (mode "L") used for the matting pass, or None
                      if refine=False.
      box          : the [x0, y0, x1, y1] box actually used, for debugging.
      grounding_score : Grounding DINO confidence, or None if a box/point was
                      passed directly.
    """
    grounding_score = None

    if isinstance(prompt, str):
        box, grounding_score = ground_text_to_box(image, prompt)
        binary_mask = sam2_binary_mask(image, box=box)
    elif isinstance(prompt, dict) and "box" in prompt:
        box = prompt["box"]
        binary_mask = sam2_binary_mask(image, box=box)
    elif isinstance(prompt, dict) and "point" in prompt:
        box = None
        binary_mask = sam2_binary_mask(
            image, point=prompt["point"], point_label=prompt.get("label", 1)
        )
    else:
        raise TypeError(
            "prompt must be a text string, {'box': [x0,y0,x1,y1]}, or "
            "{'point': [x,y]} -- got: " + repr(prompt)
        )

    if verbose:
        if grounding_score is not None:
            print(f"Grounding DINO box: {box} (score={grounding_score:.3f})")
        print(f"SAM2 mask pixels: {int(binary_mask.sum())} / {binary_mask.size}")

    if refine:
        alpha_mask, trimap = refine_with_vitmatte(image, binary_mask)
    else:
        alpha_mask, trimap = binary_mask.astype(np.float32), None

    return {
        "alpha_mask": alpha_mask,
        "binary_mask": binary_mask,
        "trimap": trimap,
        "box": box,
        "grounding_score": grounding_score,
    }

### 5 · Visual test

A quick end-to-end run on a real sample photo, showing the mask at each stage.

In [ ]:
import io
import requests
from PIL import ImageDraw

# Public-domain photo (Wikimedia Commons, "Weddingring.JPG", released into
# the public domain by uploader CLW) used purely as a real-photo smoke test
# for Stage 1 -- a plain drawn shape wouldn't meaningfully exercise either
# Grounding DINO or SAM2, both trained on real photography.
SAMPLE_IMAGE_URL = "https://commons.wikimedia.org/wiki/Special:FilePath/Weddingring.JPG"


def load_test_image() -> Image.Image:
    """Prefers a real catalogue photo you've dropped in Drive, then a public
    -domain sample photo, then a synthetic drawing as a last-resort,
    network-free fallback (see the note below on why that's a weak test).
    """
    custom_path = os.path.join(DATASET_DIR, "sample_jewellery.jpg")
    if os.path.exists(custom_path):
        print(f"Using your own test image: {custom_path}")
        return Image.open(custom_path).convert("RGB")

    try:
        resp = requests.get(
            SAMPLE_IMAGE_URL, timeout=20, headers={"User-Agent": "SkyAI-TryOn-Pipeline/1.0"}
        )
        resp.raise_for_status()
        image = Image.open(io.BytesIO(resp.content)).convert("RGB")
        image.thumbnail((1024, 1024))  # keep the smoke test fast
        print(f"Downloaded public-domain sample photo from Wikimedia Commons ({image.size[0]}x{image.size[1]}).")
        return image
    except Exception as e:
        print(f"Couldn't fetch the sample photo ({e!r}) -- falling back to a synthetic drawing.")
        print(
            "Heads up: a flat vector drawing is a weak test for Grounding DINO / SAM2, both "
            "trained on real photography -- treat this path as 'does the code run', not "
            "'does the segmentation look good'. Drop a real photo at "
            f"{custom_path} for a meaningful test."
        )
        return generate_synthetic_ring()


def generate_synthetic_ring(size: int = 800) -> Image.Image:
    """Simple ring-on-white-background, drawn locally with no network calls."""
    scale = 4  # supersample then downsize for smoother edges
    S = size * scale
    canvas = Image.new("RGB", (S, S), (245, 245, 242))
    draw = ImageDraw.Draw(canvas)

    cx, cy = S // 2, S // 2
    band_outer = int(S * 0.30)
    band_inner = int(S * 0.20)
    draw.ellipse([cx - band_outer, cy - band_outer, cx + band_outer, cy + band_outer], fill=(198, 165, 87))
    draw.ellipse([cx - band_inner, cy - band_inner, cx + band_inner, cy + band_inner], fill=(245, 245, 242))

    gem_r = int(S * 0.07)
    gem_cy = cy - band_outer
    draw.polygon(
        [(cx, gem_cy - gem_r), (cx + gem_r, gem_cy), (cx, gem_cy + gem_r), (cx - gem_r, gem_cy)],
        fill=(210, 230, 240),
    )

    return canvas.resize((size, size), Image.LANCZOS)


test_image = load_test_image()
plt.figure(figsize=(4, 4))
plt.imshow(test_image)
plt.title("Stage 1 test image")
plt.axis("off")
plt.show()

In [ ]:
# Primary demo: open-vocabulary text prompt (the production path for
# unattended catalogue ingestion).
result_text = segment_jewellery(test_image, "a diamond ring", verbose=True, refine=True)

# Secondary demo: explicit box, so the SAM2 + ViTMatte half of the pipeline
# is verified independently of Grounding DINO -- useful if you already have
# a click/crop UI, and a good way to isolate which stage is at fault if the
# text-prompt path above misfires on a given photo.
manual_box = result_text["box"]  # reuse the same region for a fair comparison
result_box = segment_jewellery(test_image, {"box": manual_box}, verbose=True, refine=True)

print("\nBoth paths produced a mask of the same shape:", result_text["alpha_mask"].shape == result_box["alpha_mask"].shape)

In [ ]:
def show_segmentation_result(image: Image.Image, result: dict, title: str = ""):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(image)
    if result["box"] is not None:
        x0, y0, x1, y1 = result["box"]
        rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="red", linewidth=2)
        axes[0].add_patch(rect)
    axes[0].set_title("Input + detected box")
    axes[0].axis("off")

    axes[1].imshow(result["binary_mask"], cmap="gray")
    axes[1].set_title("SAM2 binary mask")
    axes[1].axis("off")

    if result["trimap"] is not None:
        axes[2].imshow(result["trimap"], cmap="gray")
        axes[2].set_title("Trimap (ViTMatte input)")
    else:
        axes[2].axis("off")
    axes[2].axis("off")

    axes[3].imshow(result["alpha_mask"], cmap="gray")
    axes[3].set_title("Final alpha (ViTMatte-refined)")
    axes[3].axis("off")

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_segmentation_result(test_image, result_text, title="segment_jewellery(image, \"a diamond ring\")")
show_segmentation_result(test_image, result_box, title="segment_jewellery(image, {'box': ...})")

# Cutout-on-checkerboard is the fastest way to eyeball alpha-matte quality --
# real chain links / prong gaps will visibly show the checkerboard through
# them if the matte is working; a hard, blocky edge means it isn't.
def checkerboard(size, cell=16):
    w, h = size
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    board = ((xx // cell) + (yy // cell)) % 2 == 0
    board = board.astype(np.uint8) * 60 + 180
    return np.stack([board] * 3, axis=-1)

bg = checkerboard(test_image.size)
rgba = np.array(test_image.convert("RGB"), dtype=np.float32)
alpha = result_text["alpha_mask"][..., None]
composite = (rgba * alpha + bg * (1 - alpha)).astype(np.uint8)

plt.figure(figsize=(4, 4))
plt.imshow(composite)
plt.title("Cutout on checkerboard (alpha quality check)")
plt.axis("off")
plt.show()

## Step 1 status

**Built:** Drive mount + persistent project dir (with HF cache redirected into it) · GPU/VRAM detection with a Stage-4-aware forward note · pinned installs · SAM2.1 loaded via `transformers` (VRAM-gated size selection) · Grounding DINO for text→box grounding · ViTMatte + a `mask_to_trimap()` step it needs but the brief didn't specify · the combined `segment_jewellery(image, prompt) -> dict` function, tested on both a text prompt and an explicit box, on a real (public-domain) sample photo · pipeline state persisted to Drive.

**Known limitations, on purpose, for this step:** the sample photo is a stand-in for a real catalogue image — swap in your own at `datasets/sample_jewellery.jpg` in the project's Drive folder for a meaningful accuracy read. No fine-tuning yet (see the note above on where it plugs in). Fidelity isn't measured numerically yet — that's explicitly Step 2's job (SSIM/LPIPS/ΔE against the brief's targets).

**Things worth deciding before Step 2, if you disagree with any of my calls above:** whether the text-prompt (Grounding DINO) path is actually needed for your production flow, vs. always having an explicit box/click; and whether `-small` ViTMatte is an acceptable default or you'd rather standardize on `-base`.

Take a look, and say **"next"** when you want Step 2 (anchor detection, deterministic placement, and the fidelity validation functions).

---

# Step 2: Anchor Detection (Stage 2) + Deterministic Placement (Stage 3) + Validation Metrics

## Before you run this — what I flagged for Step 2

This step touches anthropometry and biomechanics, not just code, so there are more genuine judgment calls here than in Step 1. Going through them in the order you'll hit them below.

**1. MediaPipe Face Landmarker has no ear landmarks at all — this is the biggest gap.**
I checked the actual landmark topology (not just guessed): Face Landmarker's 478 points cover eyes, eyebrows, nose, mouth, and the face oval/jawline — there is no ear or earlobe region in the model at all, because ears sit outside what the face mesh was trained to predict. For earrings, the nearest usable points are the face-oval landmarks at the cheek/jaw edge (index 234 / 454). I checked this on a real photo (screenshot in my build notes) rather than just assuming: those points sit clearly on the face surface, well forward of where an actual earlobe hangs. I anchor earrings there with a fixed outward+downward offset to approximate the earlobe position — this is a real approximation, not a measured position, and it's the one category here most likely to need visual QA against real photos before you trust it. A dedicated ear-detection step would be the correct fix if this matters a lot to you; I didn't want to add a fourth model to Stage 2 without you weighing in.

**2. Nose-ring landmark indices: lower confidence than the others, flagged explicitly.**
For eyes, eyebrows, cheeks, and forehead, I cross-checked landmark indices against multiple independent sources and I'm confident in them. For the nose ala (nostril wing, where a nose ring actually sits), I used the commonly-referenced index pair (129 / 358), which I could not independently cross-verify from a written source the way I did the others — so I actually ran it: I loaded the real model here and checked where 129/358 land on a real, unambiguous face photo. They sit right on the nostril wings, which is reassuring, but one clean test photo isn't the same as a verified spec, so I'm leaving this flagged rather than upgrading it to "confirmed." **This is exactly why the calibration visualizer below exists**: it draws every anchor landmark this notebook uses, numbered, on your own test photo, so you can eyeball it yourself before this goes anywhere near production. If they're ever off on a different face, it's a one-line index swap.

**3. "Occlusion mask" isn't something Face/Hand Landmarker actually gives you.**
The brief's Stage 2 output list includes an occlusion mask. I checked the actual data model: `visibility`/`presence` scores exist in the landmark schema for *all* MediaPipe landmark types, but Face and Hand Landmarker leave them unset (they're not estimated) — only Pose Landmarker populates them, since occlusion-robustness is a specific feature of the BlazePose model. So: for the three pose-based categories (necklace/pendant, anklet, necklace set) you get a real per-landmark visibility score; for the six face- and hand-based categories, what I return in that field is the overall detection confidence as a named proxy, not a true occlusion signal — a hand covering part of a necklace, or hair covering an ear, won't show up here. A real occlusion mask would need a separate segmentation pass; flagging rather than quietly shipping a fake one.

**4. The biggest architectural gap: converting pixels to real millimetres needs an assumption the brief doesn't state.**
Stage 3 scales the jewellery using "physical dimensions vs. scale reference" — but physical dimensions are absolute (e.g. an 18mm ring), so the scale reference has to resolve to an absolute unit too, and a single 2D photo carries no metric information on its own (no camera calibration, no reference object). The only way to bridge that is to assume a real-world size for some measured anchor distance — e.g. "this person's inter-eye distance is ~90mm" — and derive everything else from that. I made this explicit rather than silently baking in a guess: there's an `ANTHROPOMETRIC_MM` table below of population-average real-world sizes (inter-eye distance, ring-finger width, wrist width, etc.), each used to convert a measured pixel distance into a pixels-per-mm factor for that category. **This is the single largest source of sizing error in the whole pipeline** — real people vary substantially around these averages, and shoulder width (used for necklaces) varies the most of anything here. The moment you have a better per-user signal (a stated ring size, a known height, a reference object in frame), it should replace the matching average outright.

**5. Wrist/ankle "width" from one photo isn't circumference, and bangles need circumference.**
A bangle is a closed, rigid loop — sizing it correctly needs the wrist's circumference, but a single 2D photo only gives you the *projected width* from that one viewing angle, not the actual 3D cross-section (which is elliptical, not circular). I convert width → circumference via `circumference ≈ π × width`, which assumes a circular cross-section — a real wrist is somewhat flatter than that, so this will run a bit optimistic on fit. Fine for a soft/adjustable bracelet or a watch; worth knowing about specifically for rigid bangle-stack pieces, which are the category where a wrong estimate becomes "physically doesn't fit" rather than "looks slightly off."

**6. Pose has no explicit neck point, so necklace placement is derived, not measured.**
BlazePose's 33 landmarks jump from shoulders straight to hips — there's no collarbone or neck-base point. I derive a necklace anchor as the shoulder midpoint (landmarks 11/12) plus a downward offset scaled by shoulder width, with rotation taken from the shoulder-line angle so it tilts with the body. This is a reasonable proxy, but it's worth naming as a proxy: a real necklace drapes in 3D depending on the piece's own weight and the wearer's neckline, which a 2D landmark offset can't capture.

**7. "Necklace set" shares the same anchor logic as necklace/pendant, but I'd watch it in Stage 3.** A single small pendant tolerates the rigid affine warp Stage 3 uses just fine. A large multi-tier necklace set draping across the whole collarbone is a worse fit for "one rigid scale+rotate+translate" — if Stage 3's results look wrong specifically for that category once you're testing with real pieces, this is the first place I'd look; a curved/piecewise warp along the collarbone line would be the natural upgrade, not something I've built preemptively here.

**8. The brief's own validation target has a units mismatch: "SSIM/LPIPS ≥ 0.95" doesn't work for LPIPS.**
SSIM is a similarity score (1.0 = identical, so "≥ 0.95" is a sensible bar). LPIPS is the opposite: it's a *distance* (0.0 = identical, and typical good-quality scores sit well under 0.3) — so "LPIPS ≥ 0.95" as written would mean "target near-maximum dissimilarity," which is surely not the intent. I applied the ≥0.95 threshold to SSIM only, and report LPIPS alongside it for a second, perceptual opinion without inventing a pass/fail line for it the brief never actually specified. Worth deciding what LPIPS bar you actually want (e.g. "≤ 0.10") if you'd like it to gate pass/fail too.

**10. A real finding from actually testing this, not a hypothetical: Stage 3 can legitimately fail the brief's own SSIM bar with zero model involvement, purely from resampling.**
I tested the round-trip check (item 7 above) at a few different scale factors before picking a demo value, expecting it to trivially pass every time as the brief implies ("should score high since no model touched the pixels yet"). At a mild scale change it does score high (SSIM 0.999+). But shrinking a piece to roughly a quarter of its extracted size — which is realistic: a product photo crop might be 400-800px, while the same ring rendered onto a hand in a normal portrait might only span 60-150px — measurably fails the ≥0.95 bar (I saw SSIM≈0.90) from ordinary interpolation loss on a strong downscale, before any generative model is anywhere in the picture. I checked whether interpolation choice was the culprit: switching the downscale step to `INTER_AREA` (the standard OpenCV recommendation for shrinking, vs. `INTER_LANCZOS4` which is better suited to enlarging) helps a little but doesn't close the gap. I've built both `place_jewellery()` and the fidelity checker to pick the interpolation method by direction (shrink vs. enlarge) since that's a real, if small, improvement either way — but the underlying result stands: if you see a Stage 3 FAIL on a small, heavily-downscaled piece, that's genuine information loss from resizing, not a bug in the placement code, and not something Stage 4 can fix either (harmonisation only touches pixels *around* the piece, never the piece itself). If this matters in practice, the fix would be either accepting it for small pieces (a real ring genuinely does lose some fine detail at rendered sizes that small) or extracting Stage 1 assets at a resolution closer to their expected render size instead of always maximum source resolution.

**11. Smaller calls, in case you want to override them:** rotation angle is measured as the angle (degrees, counter-clockwise from horizontal) of each category's defining axis — eye-line for the three face categories, finger-segment for the ring, wrist-to-middle-knuckle for bracelet/bangle, knee-to-ankle for the anklet, shoulder-line for the two necklace categories — documented per-function below. Running mode is `IMAGE` (single-frame, not video/live-stream) throughout, matching a try-on render pipeline rather than a live camera feed. `mediapipe`'s pip install occasionally fights with an already-installed `protobuf` version — if the install cell errors on that, the fix is a runtime restart after install, not a different package.

In [ ]:
# --- Step 2 dependencies (Stage 2: anchors, Stage 3: placement, + fidelity metrics) ---
!pip install -q -U \
    "mediapipe>=0.10.21" \
    "scikit-image>=0.24" \
    "lpips>=0.1.4"

print("Install complete.")
print("If this errored on a protobuf version conflict: Runtime > Restart session, then re-run from here.")

In [ ]:
import urllib.request

MEDIAPIPE_MODEL_DIR = os.path.join(CHECKPOINT_DIR, "mediapipe")
os.makedirs(MEDIAPIPE_MODEL_DIR, exist_ok=True)

# Official Google-hosted Tasks API bundles. "full" pose variant balances
# speed/accuracy (vs. "lite"/"heavy") -- swap the URL for either if you need to.
MEDIAPIPE_MODEL_URLS = {
    "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "pose_landmarker_full.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
}

mediapipe_model_paths = {}
for filename, url in MEDIAPIPE_MODEL_URLS.items():
    dest = os.path.join(MEDIAPIPE_MODEL_DIR, filename)
    if not os.path.exists(dest):
        print(f"Downloading {filename} ...")
        urllib.request.urlretrieve(url, dest)
    else:
        print(f"Already cached: {filename}")
    mediapipe_model_paths[filename] = dest

save_state(mediapipe_model_paths=mediapipe_model_paths)
print("\nAll MediaPipe Tasks models ready at:", MEDIAPIPE_MODEL_DIR)

In [ ]:
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions

face_landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=mediapipe_model_paths["face_landmarker.task"]),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1,
    )
)

hand_landmarker = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=mediapipe_model_paths["hand_landmarker.task"]),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2,
    )
)

pose_landmarker = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=mediapipe_model_paths["pose_landmarker_full.task"]),
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1,
    )
)

print("Face / Hand / Pose Landmarkers loaded (IMAGE mode).")
# Note: you may see a "TypeError: 'NoneType' object is not callable" during
# interpreter/session shutdown later, referencing these landmarkers' __del__.
# That's a known cosmetic mediapipe cleanup issue at process exit, not a
# functional bug -- harmless, and unrelated to anything this notebook does.

In [ ]:
# --- Landmark index constants ---
# Face indices use MediaPipe's canonical convention: "left"/"right" refer to
# the SUBJECT's own left/right (as if looking out through the photo), which
# is the opposite of viewer-left/viewer-right in a normal, non-mirrored photo.
# Confidence noted inline -- see the flagged notes above for #1 and #2.

FACE = {
    "EYE_OUTER_L": 33,        # confirmed
    "EYE_OUTER_R": 263,       # confirmed
    "EYEBROW_INNER_L": 107,   # confirmed
    "EYEBROW_INNER_R": 336,   # confirmed
    "NOSE_TIP": 4,            # confirmed
    "FOREHEAD_CENTER": 10,    # confirmed (hairline area -- less stable under hair/bangs)
    "CHEEK_EAR_PROXY_L": 234, # confirmed as a landmark, but NOT an ear point -- see flag #1
    "CHEEK_EAR_PROXY_R": 454, # ditto
    "NOSE_ALA_L": 129,        # ⚠ lower confidence -- see flag #2, verify with the calibration cell
    "NOSE_ALA_R": 358,        # ⚠ ditto
}

# Hand: standard 21-point topology, very well established, high confidence.
HAND = {
    "WRIST": 0,
    "THUMB_TIP": 4,
    "INDEX_MCP": 5, "INDEX_PIP": 6, "INDEX_TIP": 8,
    "MIDDLE_MCP": 9, "MIDDLE_TIP": 12,
    "RING_MCP": 13, "RING_PIP": 14, "RING_DIP": 15, "RING_TIP": 16,
    "PINKY_MCP": 17,
}

# Pose: standard 33-point BlazePose topology, high confidence.
POSE = {
    "LEFT_SHOULDER": 11, "RIGHT_SHOULDER": 12,
    "LEFT_HIP": 23, "RIGHT_HIP": 24,
    "LEFT_KNEE": 25, "RIGHT_KNEE": 26,
    "LEFT_ANKLE": 27, "RIGHT_ANKLE": 28,
    "LEFT_HEEL": 29, "RIGHT_HEEL": 30,
    "LEFT_FOOT_INDEX": 31, "RIGHT_FOOT_INDEX": 32,
}

# --- Anthropometric averages (mm) used to bridge pixels -> real-world size ---
# Population averages, NOT measured -- see flagged note #4. Each one pairs a
# segment that's actually directly measurable from the landmarks above with
# the population-average real-world length of that *same* segment (rather
# than inventing a conversion ratio between two unrelated measurements).
# Swap any of these for a per-user value the moment you have one (a stated
# ring size, a known height, a reference object in frame) -- that will always
# beat a population average.
ANTHROPOMETRIC_MM = {
    "interocular_width": 90.0,     # outer eye corner to outer eye corner
    "ring_finger_phalanx": 32.0,   # ring finger MCP-to-PIP segment length
    "palm_length": 95.0,           # wrist to middle-finger MCP
    "foot_length": 240.0,          # heel to foot-index (longest-toe region)
    "shoulder_width": 400.0,       # biacromial-ish width -- highest-variance assumption here
}

print("Landmark constants and anthropometric table defined.")

In [ ]:
def _run_face(image: Image.Image):
    """Returns landmark pixel coords (478x2 array), or None if no face found.
    Note: FaceLandmarkerResult carries no continuous detection score -- only
    presence/absence -- so face-based categories report confidence as 1.0/None
    rather than a graded number (see flag #3).
    """
    arr = np.array(image.convert("RGB"))
    result = face_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=arr))
    if not result.face_landmarks:
        return None
    w, h = image.size
    lms = result.face_landmarks[0]
    return np.array([[lm.x * w, lm.y * h] for lm in lms], dtype=np.float32)


def _run_hand(image: Image.Image, prefer_handedness: str = None):
    """Returns (landmarks_px (21x2), handedness_label, confidence) or (None, None, None).
    prefer_handedness: 'Left' or 'Right' (subject's hand, MediaPipe's own
    labelling) to disambiguate when both hands are visible; None = first
    detection.
    """
    arr = np.array(image.convert("RGB"))
    result = hand_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=arr))
    if not result.hand_landmarks:
        return None, None, None
    w, h = image.size
    idx = 0
    if prefer_handedness is not None:
        for i, cats in enumerate(result.handedness):
            if cats and cats[0].category_name == prefer_handedness:
                idx = i
                break
    lms = result.hand_landmarks[idx]
    px = np.array([[lm.x * w, lm.y * h] for lm in lms], dtype=np.float32)
    handedness_label = result.handedness[idx][0].category_name if result.handedness[idx] else None
    confidence = result.handedness[idx][0].score if result.handedness[idx] else None
    return px, handedness_label, confidence


def _run_pose(image: Image.Image):
    """Returns (landmarks_px (33x2), visibility (33,)) or (None, None).
    visibility is a REAL per-landmark occlusion signal here (unlike face/hand)
    -- see flag #3.
    """
    arr = np.array(image.convert("RGB"))
    result = pose_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=arr))
    if not result.pose_landmarks:
        return None, None
    w, h = image.size
    lms = result.pose_landmarks[0]
    px = np.array([[lm.x * w, lm.y * h] for lm in lms], dtype=np.float32)
    vis = np.array([lm.visibility if lm.visibility is not None else 0.0 for lm in lms], dtype=np.float32)
    return px, vis


def _anchor_result(category, anchor_xy, scale_px, scale_mm_key, rotation_deg,
                    confidence, occlusion_score, debug_points):
    """Common shape every detect_anchor_*() function returns -- see the Stage
    2 output list in the brief (anchor coords, scale reference, rotation,
    occlusion, confidence)."""
    scale_mm = ANTHROPOMETRIC_MM[scale_mm_key]
    px_per_mm = (scale_px / scale_mm) if scale_px > 0 else None
    return {
        "category": category,
        "anchor_xy": (float(anchor_xy[0]), float(anchor_xy[1])),
        "scale_reference_px": float(scale_px),
        "scale_reference_mm_assumed": scale_mm,
        "px_per_mm": px_per_mm,
        "rotation_deg": float(rotation_deg),
        "confidence": None if confidence is None else float(confidence),
        "occlusion_score": None if occlusion_score is None else float(occlusion_score),
        "debug_points": [(label, (float(p[0]), float(p[1]))) for label, p in debug_points],
    }


print("Per-family MediaPipe runners defined.")

In [ ]:
def _face_tilt_deg(px) -> float:
    """Head tilt from the eye line, in degrees -- shared rotation reference
    for all three face categories, so earrings/tikka/nose-ring all tilt
    consistently with head pose."""
    d = px[FACE["EYE_OUTER_R"]] - px[FACE["EYE_OUTER_L"]]
    return float(np.degrees(np.arctan2(d[1], d[0])))


def detect_anchor_earrings(image: Image.Image, side: str = "right") -> dict:
    """side: 'left' or 'right' earlobe, SUBJECT's own left/right (see the
    landmark-convention note above `FACE`). There is no true ear landmark in
    this model -- anchored off the nearest face-oval point with a fixed
    offset. See flag #1: treat this as the category most likely to need
    visual QA against real photos.
    """
    px = _run_face(image)
    if px is None:
        return None
    cheek_idx = FACE["CHEEK_EAR_PROXY_L"] if side == "left" else FACE["CHEEK_EAR_PROXY_R"]
    cheek_pt = px[cheek_idx]
    eye_l, eye_r = px[FACE["EYE_OUTER_L"]], px[FACE["EYE_OUTER_R"]]
    interocular_px = float(np.linalg.norm(eye_r - eye_l))

    outward = 1.0 if side == "right" else -1.0
    offset = np.array([outward * 0.12, 0.35]) * interocular_px
    anchor = cheek_pt + offset

    return _anchor_result(
        category=f"earrings_{side}", anchor_xy=anchor, scale_px=interocular_px,
        scale_mm_key="interocular_width", rotation_deg=_face_tilt_deg(px),
        confidence=1.0, occlusion_score=None,
        debug_points=[("cheek_ear_proxy", cheek_pt), ("earlobe_estimate", anchor),
                      ("eye_L", eye_l), ("eye_R", eye_r)],
    )


def detect_anchor_maang_tikka(image: Image.Image) -> dict:
    """Anchors the pendant-rest point at the glabella (derived as the
    eyebrow-inner-point midpoint -- there's no single named landmark there,
    see flag #2's neighbourhood). The hairline point is included in
    debug_points for Stage 3 to use as the chain's top attachment.
    """
    px = _run_face(image)
    if px is None:
        return None
    hairline_top = px[FACE["FOREHEAD_CENTER"]]
    brow_l, brow_r = px[FACE["EYEBROW_INNER_L"]], px[FACE["EYEBROW_INNER_R"]]
    glabella = (brow_l + brow_r) / 2.0
    eye_l, eye_r = px[FACE["EYE_OUTER_L"]], px[FACE["EYE_OUTER_R"]]
    interocular_px = float(np.linalg.norm(eye_r - eye_l))

    return _anchor_result(
        category="maang_tikka", anchor_xy=glabella, scale_px=interocular_px,
        scale_mm_key="interocular_width", rotation_deg=_face_tilt_deg(px),
        confidence=1.0, occlusion_score=None,
        debug_points=[("hairline_top", hairline_top), ("glabella_pendant_rest", glabella),
                      ("eye_L", eye_l), ("eye_R", eye_r)],
    )


def detect_anchor_nose_ring(image: Image.Image, side: str = "right") -> dict:
    """⚠ Uses the lower-confidence ala/nostril indices -- see flag #2.
    Check this one against the calibration visualizer before trusting it.
    """
    px = _run_face(image)
    if px is None:
        return None
    ala_idx = FACE["NOSE_ALA_L"] if side == "left" else FACE["NOSE_ALA_R"]
    ala_pt = px[ala_idx]
    eye_l, eye_r = px[FACE["EYE_OUTER_L"]], px[FACE["EYE_OUTER_R"]]
    interocular_px = float(np.linalg.norm(eye_r - eye_l))

    return _anchor_result(
        category=f"nose_ring_{side}", anchor_xy=ala_pt, scale_px=interocular_px,
        scale_mm_key="interocular_width", rotation_deg=_face_tilt_deg(px),
        confidence=1.0, occlusion_score=None,
        debug_points=[("nose_ala", ala_pt), ("eye_L", eye_l), ("eye_R", eye_r)],
    )


print("Face-family category functions defined: earrings, maang_tikka, nose_ring.")

In [ ]:
def _hand_tilt_deg(px, i0: int, i1: int) -> float:
    d = px[i1] - px[i0]
    return float(np.degrees(np.arctan2(d[1], d[0])))


def detect_anchor_ring(image: Image.Image, hand: str = "Right") -> dict:
    """Anchors between the ring finger's MCP and PIP joints -- the
    conventional ring-wearing position. Scale reference is that same MCP-PIP
    segment (a real measured length), paired with the average adult
    proximal-phalanx length -- not an invented width-from-length conversion.
    """
    px, handedness, confidence = _run_hand(image, prefer_handedness=hand)
    if px is None:
        return None
    mcp, pip = px[HAND["RING_MCP"]], px[HAND["RING_PIP"]]
    anchor = (mcp + pip) / 2.0
    phalanx_px = float(np.linalg.norm(pip - mcp))

    return _anchor_result(
        category="ring", anchor_xy=anchor, scale_px=phalanx_px,
        scale_mm_key="ring_finger_phalanx", rotation_deg=_hand_tilt_deg(px, HAND["RING_MCP"], HAND["RING_TIP"]),
        confidence=confidence, occlusion_score=confidence,
        debug_points=[("ring_mcp", mcp), ("ring_pip", pip), ("anchor", anchor)],
    )


def detect_anchor_bracelet_watch(image: Image.Image, hand: str = "Right") -> dict:
    """Anchors on the forearm side of the wrist landmark (a bracelet/watch
    sits proximal to the hand, not on the wrist landmark itself). Scale
    reference: wrist-to-middle-MCP ('palm length'), a real measured segment.
    See flag #5 on why this can't give a true circumference from one photo.
    """
    px, handedness, confidence = _run_hand(image, prefer_handedness=hand)
    if px is None:
        return None
    wrist, mcp = px[HAND["WRIST"]], px[HAND["MIDDLE_MCP"]]
    palm_vec = mcp - wrist
    palm_len_px = float(np.linalg.norm(palm_vec))
    direction = -palm_vec / (palm_len_px + 1e-6)
    anchor = wrist + direction * (0.35 * palm_len_px)

    return _anchor_result(
        category="bracelet_watch", anchor_xy=anchor, scale_px=palm_len_px,
        scale_mm_key="palm_length", rotation_deg=_hand_tilt_deg(px, HAND["WRIST"], HAND["MIDDLE_MCP"]),
        confidence=confidence, occlusion_score=confidence,
        debug_points=[("wrist", wrist), ("middle_mcp", mcp), ("anchor", anchor)],
    )


def detect_anchor_bangle_stack(image: Image.Image, hand: str = "Right", n_bangles: int = 3) -> dict:
    """A 'stack' is semantically several bands, not one -- the brief doesn't
    say so explicitly, but treating it identically to bracelet_watch would
    silently drop that distinction (see flag notes). Returns multiple anchors
    spaced along the wrist->forearm axis in `anchor_xy_all`; `anchor_xy` alone
    is the first (most distal) band, kept for a consistent dict shape with
    every other category.
    """
    px, handedness, confidence = _run_hand(image, prefer_handedness=hand)
    if px is None:
        return None
    wrist, mcp = px[HAND["WRIST"]], px[HAND["MIDDLE_MCP"]]
    palm_vec = mcp - wrist
    palm_len_px = float(np.linalg.norm(palm_vec))
    direction = -palm_vec / (palm_len_px + 1e-6)
    spacing = 0.12 * palm_len_px
    anchors = [wrist + direction * (0.15 * palm_len_px + i * spacing) for i in range(n_bangles)]

    result = _anchor_result(
        category="bangle_stack", anchor_xy=anchors[0], scale_px=palm_len_px,
        scale_mm_key="palm_length", rotation_deg=_hand_tilt_deg(px, HAND["WRIST"], HAND["MIDDLE_MCP"]),
        confidence=confidence, occlusion_score=confidence,
        debug_points=[("wrist", wrist), ("middle_mcp", mcp)]
                     + [(f"bangle_{i}", a) for i, a in enumerate(anchors)],
    )
    result["anchor_xy_all"] = [(float(a[0]), float(a[1])) for a in anchors]
    return result


print("Hand-family category functions defined: ring, bracelet_watch, bangle_stack.")

In [ ]:
def _pose_tilt_deg(px, i0: int, i1: int) -> float:
    d = px[i1] - px[i0]
    return float(np.degrees(np.arctan2(d[1], d[0])))


def detect_anchor_necklace_pendant(image: Image.Image) -> dict:
    """BlazePose has no neck/collarbone landmark (flag #6) -- anchor is
    derived as the shoulder midpoint plus a downward offset scaled by
    shoulder width. occlusion_score here is REAL per-landmark visibility
    (unlike the face/hand categories), since Pose actually estimates it.
    """
    px, vis = _run_pose(image)
    if px is None:
        return None
    l_sh, r_sh = px[POSE["LEFT_SHOULDER"]], px[POSE["RIGHT_SHOULDER"]]
    mid = (l_sh + r_sh) / 2.0
    shoulder_px = float(np.linalg.norm(r_sh - l_sh))
    anchor = mid + np.array([0.0, 0.28 * shoulder_px])
    occlusion = float((vis[POSE["LEFT_SHOULDER"]] + vis[POSE["RIGHT_SHOULDER"]]) / 2.0)

    return _anchor_result(
        category="necklace_pendant", anchor_xy=anchor, scale_px=shoulder_px,
        scale_mm_key="shoulder_width", rotation_deg=_pose_tilt_deg(px, POSE["LEFT_SHOULDER"], POSE["RIGHT_SHOULDER"]),
        confidence=occlusion, occlusion_score=occlusion,
        debug_points=[("left_shoulder", l_sh), ("right_shoulder", r_sh), ("neck_estimate", anchor)],
    )


def detect_anchor_necklace_set(image: Image.Image) -> dict:
    """Same anchor family as necklace_pendant -- see flag #7 on why a large
    multi-tier piece may outgrow Stage 3's rigid single-anchor warp; nothing
    upstream here changes for that, it's purely a Stage 3 concern to watch.
    """
    result = detect_anchor_necklace_pendant(image)
    if result is not None:
        result["category"] = "necklace_set"
    return result


def detect_anchor_anklet(image: Image.Image, side: str = "right") -> dict:
    px, vis = _run_pose(image)
    if px is None:
        return None
    ankle_idx = POSE["LEFT_ANKLE"] if side == "left" else POSE["RIGHT_ANKLE"]
    heel_idx = POSE["LEFT_HEEL"] if side == "left" else POSE["RIGHT_HEEL"]
    foot_idx = POSE["LEFT_FOOT_INDEX"] if side == "left" else POSE["RIGHT_FOOT_INDEX"]
    knee_idx = POSE["LEFT_KNEE"] if side == "left" else POSE["RIGHT_KNEE"]

    ankle, heel, foot = px[ankle_idx], px[heel_idx], px[foot_idx]
    foot_len_px = float(np.linalg.norm(foot - heel))
    occlusion = float(vis[ankle_idx])

    return _anchor_result(
        category=f"anklet_{side}", anchor_xy=ankle, scale_px=foot_len_px,
        scale_mm_key="foot_length", rotation_deg=_pose_tilt_deg(px, knee_idx, ankle_idx),
        confidence=occlusion, occlusion_score=occlusion,
        debug_points=[("ankle", ankle), ("heel", heel), ("foot_index", foot)],
    )


print("Pose-family category functions defined: necklace_pendant, necklace_set, anklet.")

In [ ]:
CATEGORY_FUNCTIONS = {
    "earrings": detect_anchor_earrings,
    "maang_tikka": detect_anchor_maang_tikka,
    "nose_ring": detect_anchor_nose_ring,
    "ring": detect_anchor_ring,
    "bracelet_watch": detect_anchor_bracelet_watch,
    "bangle_stack": detect_anchor_bangle_stack,
    "necklace_pendant": detect_anchor_necklace_pendant,
    "necklace_set": detect_anchor_necklace_set,
    "anklet": detect_anchor_anklet,
}


def detect_anchor(image: Image.Image, category: str, **kwargs) -> dict:
    """Single entry point for all 9 categories -- Stage 2's deliverable.
    Returns None if the relevant landmarker found nothing in the image.
    """
    if category not in CATEGORY_FUNCTIONS:
        raise ValueError(f"Unknown category '{category}'. Options: {list(CATEGORY_FUNCTIONS)}")
    return CATEGORY_FUNCTIONS[category](image, **kwargs)


print("detect_anchor(image, category) ready. Categories:", list(CATEGORY_FUNCTIONS))

In [ ]:
def visualize_anchor_calibration(image: Image.Image, results, title: str = "Anchor calibration check"):
    """Draws every debug landmark from one or more detect_anchor(...) results
    onto the actual image, labelled. This is the direct answer to "tell me
    rather than guessing silently" for the landmarks I flagged as
    lower-confidence (nose ala) or approximate by construction (earlobe,
    neck) -- look at the dots on your own photo before trusting a category.
    """
    if isinstance(results, dict):
        results = [results]
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)
    for result in results:
        if result is None:
            continue
        for label, (x, y) in result["debug_points"]:
            ax.plot(x, y, "o", markersize=5)
            ax.annotate(label, (x, y), fontsize=8, color="red", xytext=(4, 4), textcoords="offset points")
        ax.plot(*result["anchor_xy"], "r+", markersize=15, markeredgewidth=2)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### 6 · Stage 3 — deterministic geometric placement (pure OpenCV, no model calls)

In [ ]:
def place_jewellery(portrait_rgb: np.ndarray, asset_rgb: np.ndarray, asset_alpha: np.ndarray,
                     anchor_xy, px_per_mm: float, physical_size_mm: float,
                     asset_measurement_px: float, rotation_deg: float):
    """Stage 3: scale + rotate + translate the Stage 1 asset onto the Stage 2
    anchor. Pure cv2.warpAffine -- no model calls, matching the brief exactly.

    portrait_rgb          : HxWx3 uint8, the wearer photo.
    asset_rgb, asset_alpha: the Stage 1 segmented piece (RGB + float32 alpha
                             in [0,1]), at its native extracted resolution.
    anchor_xy              : (x, y) in portrait pixel coords -- from Stage 2.
    px_per_mm              : from Stage 2's scale reference (see flag #4).
    physical_size_mm        : the piece's own known real-world size (catalogue
                             metadata -- e.g. a ring's outer diameter).
    asset_measurement_px    : the SAME measurement as physical_size_mm, but in
                             pixels, on the extracted asset as it stands
                             (e.g. that ring's outer diameter in the Stage 1
                             crop) -- this is what makes the scale factor
                             correct rather than just "scale to fit an anchor
                             box".
    rotation_deg            : from Stage 2.

    Returns: (composite, warped_rgb, warped_alpha, scale_factor)
    """
    target_px = physical_size_mm * px_per_mm
    scale_factor = target_px / asset_measurement_px

    h_p, w_p = portrait_rgb.shape[:2]
    h_a, w_a = asset_alpha.shape[:2]
    asset_center = (w_a / 2.0, h_a / 2.0)

    M = cv2.getRotationMatrix2D(asset_center, rotation_deg, scale_factor)
    M[:, 2] += np.array([anchor_xy[0] - asset_center[0], anchor_xy[1] - asset_center[1]])

    # INTER_AREA is the standard OpenCV choice when shrinking (avoids moire/
    # aliasing); INTER_LANCZOS4 when enlarging or scale ~1. See flag #10 --
    # tested this against the alternative and it measurably matters for
    # small pieces placed at a much lower resolution than they were extracted at.
    interp = cv2.INTER_AREA if scale_factor < 1.0 else cv2.INTER_LANCZOS4

    warped_rgb = cv2.warpAffine(
        asset_rgb, M, (w_p, h_p), flags=interp,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0),
    )
    warped_alpha = cv2.warpAffine(
        asset_alpha, M, (w_p, h_p), flags=interp,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )

    a = warped_alpha[..., None]
    composite = (warped_rgb.astype(np.float32) * a + portrait_rgb.astype(np.float32) * (1 - a)).astype(np.uint8)
    return composite, warped_rgb, warped_alpha, scale_factor


print("place_jewellery() defined (Stage 3).")

### 7 · Fidelity validation functions

SSIM + LPIPS + ΔE, checked with a **round-trip warp**: forward-transform the Stage 1 asset with Stage 3's own scale+rotate, then inverse-transform it back, and compare to the untouched original. This isolates whether Stage 3's transform itself preserves the piece — exactly the "should score high since no model touched the pixels yet" sanity check the brief asks for, and a real (if strict) test rather than a tautology, since resampling twice does introduce a small amount of interpolation blur.

In [ ]:
import lpips
from skimage.metrics import structural_similarity as ssim
from skimage.color import rgb2lab, deltaE_ciede2000

lpips_fn = lpips.LPIPS(net="alex").to(DEVICE)
lpips_fn.eval()
print(f"LPIPS (AlexNet backbone) loaded on {DEVICE}.")

In [ ]:
def compute_lpips(img1_uint8: np.ndarray, img2_uint8: np.ndarray) -> float:
    def to_tensor(img):
        t = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 127.5 - 1.0
        return t.to(DEVICE)

    with torch.no_grad():
        d = lpips_fn(to_tensor(img1_uint8), to_tensor(img2_uint8))
    return float(d.item())


def _to3x3(M):
    return np.vstack([M, [0, 0, 1]])


def fidelity_round_trip_check(asset_rgb: np.ndarray, asset_alpha: np.ndarray,
                               scale_factor: float, rotation_deg: float, pad_ratio: float = 0.6) -> dict:
    """Forward-warp the asset with Stage 3's own scale+rotate, then
    inverse-warp back, and compare to the untouched original -- see the
    markdown note above on why this (rather than comparing against the
    composite) is the fair test of Stage 3's transform fidelity.
    """
    h, w = asset_alpha.shape
    pad = int(max(h, w) * pad_ratio)
    H, W = h + 2 * pad, w + 2 * pad
    center_src = (w / 2.0, h / 2.0)
    center_pad = (W / 2.0, H / 2.0)

    M_fwd = cv2.getRotationMatrix2D(center_pad, rotation_deg, scale_factor)
    shift = np.array([[1, 0, center_pad[0] - center_src[0]],
                       [0, 1, center_pad[1] - center_src[1]]], dtype=np.float64)
    full_fwd = (_to3x3(M_fwd) @ _to3x3(shift))[:2, :]
    # Same direction-aware interpolation choice as place_jewellery(), applied
    # to each leg of the round trip independently (forward and inverse scale
    # in opposite directions).
    interp_fwd = cv2.INTER_AREA if scale_factor < 1.0 else cv2.INTER_LANCZOS4
    interp_inv = cv2.INTER_AREA if (1.0 / scale_factor) < 1.0 else cv2.INTER_LANCZOS4

    warped_rgb = cv2.warpAffine(asset_rgb, full_fwd, (W, H), flags=interp_fwd,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    warped_alpha = cv2.warpAffine(asset_alpha, full_fwd, (W, H), flags=interp_fwd,
                                   borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    full_inv = np.linalg.inv(_to3x3(full_fwd))[:2, :]
    back_rgb = cv2.warpAffine(warped_rgb, full_inv, (w, h), flags=interp_inv,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    back_alpha = cv2.warpAffine(warped_alpha, full_inv, (w, h), flags=interp_inv,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    m = asset_alpha > 0.5
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    orig_crop = asset_rgb[y0:y1, x0:x1]
    back_crop = np.clip(back_rgb[y0:y1, x0:x1], 0, 255).astype(np.uint8)
    crop_mask = m[y0:y1, x0:x1]

    ssim_val = ssim(orig_crop, back_crop, channel_axis=2, data_range=255)

    lab1 = rgb2lab(orig_crop.astype(np.float64) / 255.0)
    lab2 = rgb2lab(back_crop.astype(np.float64) / 255.0)
    de_map = deltaE_ciede2000(lab1, lab2)
    delta_e_mean = float(de_map[crop_mask].mean())

    lpips_val = compute_lpips(orig_crop, back_crop)

    return {
        "ssim": float(ssim_val),
        "delta_e": delta_e_mean,
        "lpips": lpips_val,
        "back_rgb": back_rgb,
        "back_alpha": back_alpha,
    }


# Brief's stated targets -- see flag #8 on the SSIM-vs-LPIPS units mismatch.
FIDELITY_TARGETS = {"ssim_min": 0.95, "delta_e_max": 2.0}


def print_fidelity_report(metrics: dict, label: str = "") -> bool:
    ssim_pass = metrics["ssim"] >= FIDELITY_TARGETS["ssim_min"]
    de_pass = metrics["delta_e"] <= FIDELITY_TARGETS["delta_e_max"]
    print(f"--- Fidelity report {label} ---")
    print(f"SSIM    : {metrics['ssim']:.4f}  (target >= {FIDELITY_TARGETS['ssim_min']})  "
          f"{'PASS' if ssim_pass else 'FAIL'}")
    print(f"LPIPS   : {metrics['lpips']:.4f}  (lower is better; no brief-stated threshold, see flag #8)")
    print(f"Delta E : {metrics['delta_e']:.4f}  (target <= {FIDELITY_TARGETS['delta_e_max']})  "
          f"{'PASS' if de_pass else 'FAIL'}")
    overall = ssim_pass and de_pass
    print(f"Overall : {'PASS' if overall else 'FAIL'}")
    return overall


print("Fidelity validation functions ready.")

### 8 · End-to-end: Stages 1 → 2 → 3 on a real photo pair

Ring (Stage 1's jewellery photo) placed onto a hand (a new, real portrait photo) — chosen because it's the category with the least landmark ambiguity, so this demo tests the *pipeline wiring*, not the anchor-approximation judgment calls flagged above. The same `detect_anchor(image, category)` call works identically for all 9 categories.

In [ ]:
HAND_SAMPLE_URL = "https://commons.wikimedia.org/wiki/Special:FilePath/Hand_-_Fingers.jpg"
# Public-domain photo (Wikimedia Commons, "Hand - Fingers.jpg", released into
# the public domain by uploader Coolgirly88) -- same reasoning as Step 1's
# ring photo: a real hand photo is the only meaningful test for Hand
# Landmarker, a synthetic drawing won't be detected reliably.


def generate_synthetic_hand(size: int = 800) -> Image.Image:
    """Rough hand-shaped silhouette, network-free fallback. Flagging clearly:
    MediaPipe's Hand Landmarker very likely will NOT detect anything on this
    -- it exists only so the code path doesn't crash without internet, not as
    a meaningful test. Same caveat as Step 1's synthetic ring.
    """
    canvas = Image.new("RGB", (size, size), (235, 220, 200))
    draw = ImageDraw.Draw(canvas)
    draw.ellipse([size * 0.30, size * 0.45, size * 0.70, size * 0.85], fill=(210, 170, 140))
    finger_w = size * 0.07
    for fx in (0.35, 0.45, 0.55, 0.65):
        draw.rounded_rectangle(
            [size * fx - finger_w / 2, size * 0.15, size * fx + finger_w / 2, size * 0.5],
            radius=finger_w / 2, fill=(210, 170, 140),
        )
    draw.rounded_rectangle(
        [size * 0.18, size * 0.5, size * 0.18 + finger_w * 1.2, size * 0.7],
        radius=finger_w * 0.6, fill=(210, 170, 140),
    )
    return canvas


def load_hand_test_image() -> Image.Image:
    custom_path = os.path.join(DATASET_DIR, "sample_hand.jpg")
    if os.path.exists(custom_path):
        print(f"Using your own hand test image: {custom_path}")
        return Image.open(custom_path).convert("RGB")

    try:
        resp = requests.get(
            HAND_SAMPLE_URL, timeout=20, headers={"User-Agent": "SkyAI-TryOn-Pipeline/1.0"}
        )
        resp.raise_for_status()
        image = Image.open(io.BytesIO(resp.content)).convert("RGB")
        image.thumbnail((1024, 1024))
        print(f"Downloaded public-domain hand photo from Wikimedia Commons ({image.size[0]}x{image.size[1]}).")
        return image
    except Exception as e:
        print(f"Couldn't fetch the sample hand photo ({e!r}) -- falling back to a synthetic silhouette.")
        print(f"Drop a real hand photo at {custom_path} for a meaningful test.")
        return generate_synthetic_hand()


hand_image = load_hand_test_image()
plt.figure(figsize=(4, 4))
plt.imshow(hand_image)
plt.title("Stage 2/3 test portrait (hand)")
plt.axis("off")
plt.show()

In [ ]:
ring_anchor = detect_anchor(hand_image, "ring", hand="Right")

if ring_anchor is None:
    print(
        "No hand detected in this photo. If you're on the synthetic fallback image, "
        "that's expected (see the note above) -- drop a real hand photo at "
        f"{os.path.join(DATASET_DIR, 'sample_hand.jpg')} and re-run from the previous cell."
    )
else:
    print(f"anchor_xy            : {ring_anchor['anchor_xy']}")
    print(f"scale_reference_px    : {ring_anchor['scale_reference_px']:.1f}")
    print(f"px_per_mm             : {ring_anchor['px_per_mm']:.3f}  (assumes {ring_anchor['scale_reference_mm_assumed']}mm -- see flag #4)")
    print(f"rotation_deg          : {ring_anchor['rotation_deg']:.1f}")
    print(f"confidence            : {ring_anchor['confidence']}")

    visualize_anchor_calibration(hand_image, ring_anchor, title="Ring anchor -- verify against the photo")

In [ ]:
def crop_to_mask(rgb: np.ndarray, alpha: np.ndarray, pad_frac: float = 0.05):
    """Tight-crop the Stage 1 output to its alpha mask -- Stage 3 wants the
    jewellery asset alone, not the whole (mostly-empty) source photo canvas.
    """
    m = alpha > 0.02
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    h, w = y1 - y0, x1 - x0
    pad_y, pad_x = int(h * pad_frac), int(w * pad_frac)
    y0, y1 = max(0, y0 - pad_y), min(rgb.shape[0], y1 + pad_y)
    x0, x1 = max(0, x0 - pad_x), min(rgb.shape[1], x1 + pad_x)
    return rgb[y0:y1, x0:x1], alpha[y0:y1, x0:x1]


if ring_anchor is not None:
    # Stage 1 output, from Step 1's segment_jewellery() on the ring photo.
    asset_rgb_full = np.array(test_image.convert("RGB"))
    asset_alpha_full = result_text["alpha_mask"]
    asset_rgb, asset_alpha = crop_to_mask(asset_rgb_full, asset_alpha_full)

    # Catalogue metadata this piece would carry in production; the asset's
    # own bounding-box extent stands in for "measure the same thing in the
    # extracted crop" (see place_jewellery()'s docstring on why both are needed).
    physical_size_mm = 20.0
    asset_measurement_px = float(max(asset_alpha.shape))

    portrait_rgb = np.array(hand_image.convert("RGB"))
    composite, warped_rgb, warped_alpha, scale_factor = place_jewellery(
        portrait_rgb, asset_rgb, asset_alpha,
        anchor_xy=ring_anchor["anchor_xy"],
        px_per_mm=ring_anchor["px_per_mm"],
        physical_size_mm=physical_size_mm,
        asset_measurement_px=asset_measurement_px,
        rotation_deg=ring_anchor["rotation_deg"],
    )
    print(f"Computed scale_factor: {scale_factor:.4f}  "
          f"(target size {physical_size_mm}mm x {ring_anchor['px_per_mm']:.3f}px/mm "
          f"= {physical_size_mm * ring_anchor['px_per_mm']:.1f}px, "
          f"asset was {asset_measurement_px:.0f}px)")

    metrics = fidelity_round_trip_check(asset_rgb, asset_alpha, scale_factor, ring_anchor["rotation_deg"])
    passed = print_fidelity_report(metrics, label="(Stages 1-3, hard-pasted, no harmonisation yet)")

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(composite)
    axes[0].scatter(*ring_anchor["anchor_xy"], c="red", marker="+", s=200)
    axes[0].set_title("Hard-pasted composite (Stages 1-3)")
    axes[0].axis("off")
    axes[1].imshow(asset_rgb)
    axes[1].set_title("Original Stage 1 asset (for comparison)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    print(
        "\nNo shadow/lighting integration yet by design -- that's Stage 4, "
        "Step 3's job. This is the flat 'hard-pasted' composite the brief "
        "describes as Stage 3's actual output."
    )

## Step 2 status

**Built:** MediaPipe Face/Hand/Pose Landmarkers (downloaded once, cached to Drive) · landmark constants with confidence levels noted per index · an anthropometric mm table bridging pixels to real-world size, with the assumption made explicit rather than hidden · all 9 category functions (earrings, maang_tikka, nose_ring, ring, bracelet_watch, bangle_stack, necklace_pendant, necklace_set, anklet) behind one `detect_anchor(image, category)` dispatcher · a calibration visualizer for checking any anchor against a real photo · Stage 3's `place_jewellery()` (pure OpenCV, direction-aware interpolation) · fidelity validation (`fidelity_round_trip_check`) against the brief's SSIM/ΔE targets, with LPIPS reported alongside · a real end-to-end run: the Step 1 ring asset placed onto a real (public-domain) hand photo, with metrics.

**What I verified by actually testing, not just reasoning about:** loaded the real Face Landmarker model here and confirmed all ten face landmark indices land exactly where expected on a real photo (screenshot in my build notes) — including the cheek/jaw "ear proxy" points visibly *not* being on the ear, and the nose-ala pair landing correctly on the nostril wings. Confirmed the exact MediaPipe Tasks API surface (constructors, result field names, handedness/visibility fields) by loading the real installed library and running Face and Hand Landmarker end-to-end. Tested the warp math, SSIM, and ΔE with real `opencv`/`scikit-image` calls, which is how I caught two real bugs before you'd ever see them: a mislabeled "checkerboard" that was actually diagonal stripes, and the aggressive-downscale SSIM-failure finding in flag #10. Pose Landmarker and LPIPS are built against the same well-established, stable APIs but weren't run live in my build environment (no pose model file or `torch` available there) — worth a slightly closer look the first time you run those specific cells.

**Known limitations, on purpose, for this step:** anchor precision depends on the anthropometric averages in flag #4 — real per-user sizing will beat this the moment you have it. Earrings and nose ring are approximations off the nearest available landmark, not measurements of the actual feature (flags #1, #2) — use the calibration visualizer before trusting them on new photos. Only the `ring` category is exercised in the live end-to-end demo; the other 8 use the same `detect_anchor()` call but aren't individually demonstrated here.

Take a look — say **"next"** for Step 3 (bounded generative harmonisation, LoRA training, and the full 4-stage demo).

---

# Step 3: Bounded Harmonisation (Stage 4) + LoRA Training + Full Demo

## Before you run this — what I flagged for Step 3

This step involved the most research of the three, because two of its central assumptions don't survive contact with how these specific models actually work today. Going through everything in order.

**1. FLUX.1 Fill [dev]'s license is non-commercial — this matters a lot for SkyAI specifically.**
I checked the actual license text rather than assuming: FLUX.1 [dev] Non-Commercial License v1.1.1 permits personal/research use and evaluation by commercial entities, but explicitly excludes production use — "so far as you do not receive any direct or indirect payment arising from the use of the FLUX.1 [dev] Model." SkyAI is a commercial SaaS selling a try-on product. That means **FLUX.1 Fill [dev] cannot legally be the production harmonisation backbone**, full stop — this isn't a "check with legal later" footnote, it's disqualifying for this specific deployment shape as described. The brief's own "deployment/licensing out of scope for this phase" framing doesn't make this go away; it's a model-selection fact, not a deployment detail. Your two real options: FLUX.1 Fill **[pro]** via Black Forest Labs' paid API (commercial terms, but you lose self-hosting and pay per-call), or lean on the Qwen backbone below, which has no such restriction. I've built and left FLUX.1 Fill [dev] in this notebook because you asked for it and it's genuinely useful for **prototyping and quality comparison** — just don't let it quietly become the production path.

**2. Qwen-Image-Edit-2511 doesn't support masked inpainting the way FLUX.1 Fill does — "the same masking logic" for both backbones isn't literally achievable.**
I checked the model's actual, officially-documented pipeline (`QwenImageEditPlusPipeline`): it takes a full image and a text instruction and edits the *whole* image — there is no `mask_image` parameter. This is a known, openly-discussed limitation (I found a Hugging Face thread where someone asks exactly this: how do you guarantee the main subject doesn't change, since there's no mask?). There is a separate, community-contributed `QwenImageEditInpaintPipeline` that does take a mask, but every example I could find for it targets the base `Qwen-Image-Edit` checkpoint, not the newer "-2511"/"Plus" variant the brief names — I couldn't confirm compatibility, and didn't want to ship a confident-looking code path built on an unverified assumption.

So the two backbones protect the jewellery pixels through genuinely different mechanisms here, not "the same masking logic": FLUX.1 Fill gets a hard pixel mask (a structural guarantee — the model architecturally cannot touch masked-out pixels). Qwen gets an explicit instruction telling it what to change and what to leave alone, **plus** a mandatory post-generation step that pastes the original Stage 1 jewellery pixels back on top of whatever Qwen produced, using the same alpha mask. That second step is what makes the fidelity guarantee real rather than hopeful — without it, "don't change the ring" is a request, not a constraint. Both paths end up protecting the piece equally well for the fidelity metrics below; they just get there differently, and the harmonisation quality *outside* the piece is genuinely what you're comparing, which is what the brief actually wants A/B'd.

**3. The LoRA training config for FLUX.1 Fill specifically isn't something I could verify — flagging rather than guessing.**
The brief's `control_path`/`folder_path` pattern is real — I found and confirmed it against ai-toolkit's actual example config for **FLUX.1 Kontext** (an edit-style Flux variant), which is the closest verified analogue. But Kontext and Fill are architecturally different Flux variants (Kontext takes a reference image + instruction; Fill takes an image + mask), and I could not find an official ai-toolkit example config specifically for training a Fill LoRA. I've written the YAML generator below using the confirmed Kontext structure with `arch: "flux_fill"` as my best-guess model-family string — **that specific string is not verified**. Before running the FLUX training path for real, check ai-toolkit's current `config/examples/` folder on GitHub for a Fill-specific example and correct the `arch` value if needed.

**4. The Qwen LoRA config has a different, better-documented risk: a known bug report on exactly this workflow.**
Unlike Fill, ai-toolkit does ship official example configs for Qwen Image Edit (`train_lora_qwen_image_edit_32gb.yaml`, `train_lora_qwen_image_edit_2509_32gb.yaml`) — so the general path is real and supported. But I also found an open GitHub issue titled "Missing control images for QwenImageEditPlusModel," reporting exactly the paired-image (reference/target) training setup this brief needs, on the "Plus" architecture family that Qwen-Image-Edit-2511 uses. I don't know if it's since been fixed. I've built the config generator to match ai-toolkit's confirmed schema as closely as I can, but **check that issue and the current example config before trusting this path for anything beyond a quick smoke test.**

**5. GPU tier (flagged back in Step 1) is no longer a future concern — it's the whole ballgame here.**
Both backbones are large: FLUX.1 Fill [dev] is a ~12B-parameter model, and Qwen-Image-Edit is in a similar or larger class. On anything short of an A100, both inference and training need real memory tricks (4-bit/8-bit quantization, CPU offload, gradient checkpointing) just to run at all, separate from whether they run *well*. I've defaulted to `bfloat16` plus `enable_model_cpu_offload()` for inference and `quantize: true` + `adamw8bit` for training as reasonable defaults, but if you're on a T4, expect this step to be slow and possibly memory-constrained regardless.

**6. `ai-toolkit` isn't a pip package you import — it's a repo you clone and run as a subprocess.**
There's no `import ai_toolkit; train(...)` API. The real workflow is: clone the repo, write a YAML config to disk, then shell out to `python run.py your_config.yaml`. I've built it that way rather than pretending there's a cleaner Python API, since presenting it otherwise would just be wrong.

**7. Checkpoint-resume after a disconnect: implemented based on how this class of trainer conventionally behaves, not independently verified this session.**
Re-running the same config, pointed at the same Drive-persisted `training_folder`, is how ai-toolkit is designed to be resumed (it looks for the latest checkpoint under that name and continues). I've built the training cell to be safely re-run for exactly this reason. I didn't run a real multi-hour training job to confirm the resume behaviour end-to-end — if it doesn't pick up where it left off, that's the first thing to check against ai-toolkit's current docs.

**8. Placeholder training pairs are synthetic and honestly weak — treat the LoRA training section as a wiring test, not a quality signal.**
Since we have no real before/after retouch pairs, I generate them by taking Stage 3's flat hard-paste and adding a crude synthetic drop-shadow ellipse as the "after." A LoRA trained on this will learn "add a generic dark smudge near jewellery," not real contact-shadow physics — it exists so the training code path runs and produces a loadable checkpoint, not to produce a LoRA worth using. The real dataset (50-150 professionally-retouched pairs, per the brief) is what actually needs to exist before this step means anything quality-wise.

In [ ]:
# --- Step 3 dependencies (Stage 4: harmonisation + LoRA training) ---
# diffusers from git: Qwen-Image-Edit-2511's own model card recommends this
# (its pipeline landed very recently relative to diffusers' stable releases
# at the time); pinning to git head is the documented way to guarantee support.
!pip install -q -U "git+https://github.com/huggingface/diffusers"
!pip install -q -U \
    "transformers>=4.56" \
    "accelerate>=0.34" \
    "peft>=0.13" \
    "bitsandbytes>=0.43" \
    "sentencepiece>=0.2" \
    "protobuf>=4.25"

print("Install complete.")
print("If Qwen's pipeline import fails: Runtime > Restart session, then re-run from here")
print("(a fresh git install of diffusers sometimes needs a clean process to pick up new pipeline classes).")

### 9 · Stage 4 — bounded generative harmonisation

The only generative stage. Builds a boundary mask (dilate the Stage 1/3 mask outward, subtract the original) so harmonisation can only touch the ring of pixels around the piece — never the piece itself.

In [ ]:
HARMONISE_PROMPT = "harmonize lighting and add realistic contact shadows to the jewelry"
# Single, consistent instruction phrase -- matches the brief's captioning
# guidance for the LoRA dataset, and doubles as the inference-time prompt for
# both backbones so they're being asked to do the same thing.


def build_harmonisation_mask(portrait_alpha: np.ndarray, dilate_px: int = 25) -> Image.Image:
    """Dilate the (portrait-frame) jewellery mask outward, subtract the
    original -- the resulting ring is the ONLY region either backbone is
    allowed to touch. White = inpaint, black = keep untouched, matching
    standard diffusers mask convention.
    """
    binary = (portrait_alpha > 0.5).astype(np.uint8) * 255
    kernel = np.ones((dilate_px, dilate_px), np.uint8)
    dilated = cv2.dilate(binary, kernel, iterations=1)
    ring = cv2.subtract(dilated, binary)
    return Image.fromarray(ring, mode="L")


print("Harmonisation mask builder ready.")

In [ ]:
from diffusers import FluxFillPipeline

# ⚠ LICENSE: FLUX.1 [dev] Non-Commercial License v1.1.1 -- non-commercial /
# non-production use only. See flag #1 above: this cannot legally be the
# production backbone for a commercial SaaS as described. Also gated on HF --
# accept the license at the model page and authenticate (huggingface-cli
# login / HF_TOKEN) before this cell will download successfully.
FLUX_FILL_REPO = "black-forest-labs/FLUX.1-Fill-dev"

flux_fill_pipe = FluxFillPipeline.from_pretrained(FLUX_FILL_REPO, torch_dtype=torch.bfloat16)
flux_fill_pipe.enable_model_cpu_offload()  # trade speed for VRAM headroom -- see flag #5
print("FLUX.1 Fill [dev] loaded -- NON-COMMERCIAL LICENSE, see flag #1.")


def harmonise_flux(portrait_rgb: np.ndarray, portrait_alpha: np.ndarray, dilate_px: int = 25,
                    guidance_scale: float = 30.0, num_inference_steps: int = 50, seed: int = 0) -> Image.Image:
    """Stage 4, FLUX.1 Fill backbone. guidance_scale=30 matches the model's
    own documented default (Fill uses a different guidance convention to
    regular FLUX.1-dev's ~3.5 -- this is not a typo)."""
    h, w = portrait_rgb.shape[:2]
    h16, w16 = (h // 16) * 16, (w // 16) * 16  # Flux wants multiples of 16
    image = Image.fromarray(portrait_rgb).resize((w16, h16))
    mask = build_harmonisation_mask(portrait_alpha, dilate_px).resize((w16, h16))

    result = flux_fill_pipe(
        prompt=HARMONISE_PROMPT,
        image=image,
        mask_image=mask,
        height=h16, width=w16,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        max_sequence_length=512,
        generator=torch.Generator("cpu").manual_seed(seed),
    ).images[0]
    return result.resize((w, h))


print("harmonise_flux() ready.")

In [ ]:
from diffusers import QwenImageEditPlusPipeline

# License: Apache 2.0 -- no commercial restriction (contrast with flag #1).
QWEN_EDIT_REPO = "Qwen/Qwen-Image-Edit-2511"

qwen_edit_pipe = QwenImageEditPlusPipeline.from_pretrained(QWEN_EDIT_REPO, torch_dtype=torch.bfloat16)
qwen_edit_pipe.enable_model_cpu_offload()
print("Qwen-Image-Edit-2511 loaded -- Apache 2.0 license.")


def harmonise_qwen(portrait_rgb: np.ndarray, portrait_alpha: np.ndarray,
                    guidance_scale: float = 1.0, true_cfg_scale: float = 4.0,
                    num_inference_steps: int = 40, seed: int = 0) -> Image.Image:
    """Stage 4, Qwen backbone. No native mask_image support (see flag #2) --
    the prompt explicitly states what to preserve, and the original Stage 1
    jewellery pixels are pasted back on top afterward using the SAME alpha
    mask. That restore step, not the prompt, is what actually guarantees
    fidelity here.
    """
    image = Image.fromarray(portrait_rgb)
    prompt = (
        f"{HARMONISE_PROMPT}. Keep the jewelry itself completely unchanged -- "
        f"same shape, same gemstones, same metal color and finish -- only add "
        f"a soft contact shadow and match the ambient lighting around it."
    )
    edited = qwen_edit_pipe(
        image=[image],
        prompt=prompt,
        negative_prompt=" ",
        true_cfg_scale=true_cfg_scale,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=torch.manual_seed(seed),
    ).images[0]
    edited = edited.resize(image.size)

    # Mandatory fidelity restore -- see flag #2. Without this, "keep the
    # jewelry unchanged" is a request the model can ignore, not a constraint.
    edited_arr = np.array(edited.convert("RGB"), dtype=np.float32)
    original_arr = portrait_rgb.astype(np.float32)
    a = portrait_alpha[..., None]
    restored = (original_arr * a + edited_arr * (1 - a)).astype(np.uint8)
    return Image.fromarray(restored)


print("harmonise_qwen() ready.")

### 10 · LoRA fine-tuning (`ostris/ai-toolkit`)

Trains a small adapter on top of the chosen base model so it acts strictly as a lighting/shadow compositor, not a general-purpose generator — per the brief's dataset spec (paired hard-paste "reference" / retouched "target" images). See flags #3, #4, #6, #7, #8 above before trusting this section beyond a wiring test.

In [ ]:
def generate_placeholder_training_pair(size: int = 512, seed: int = 0):
    """Synthetic (hard-paste 'reference', retouched 'target') pair -- see
    flag #8. This exercises the training CODE PATH only; a LoRA trained on
    these learns 'add a generic dark smudge', not real shadow physics. Real
    data (50-150 professionally-retouched pairs, per the brief) is what
    actually needs to exist before this section means anything quality-wise.
    """
    rng = np.random.RandomState(seed)
    bg_val = int(rng.randint(180, 230))
    canvas = np.full((size, size, 3), bg_val, dtype=np.uint8)

    cx = size // 2 + int(rng.randint(-30, 30))
    cy = size // 2 + int(rng.randint(-30, 30))
    r_outer = int(rng.randint(50, 70))
    r_inner = int(r_outer * 0.65)

    before = canvas.copy()
    yy, xx = np.ogrid[:size, :size]
    dist2 = (xx - cx) ** 2 + (yy - cy) ** 2
    ring_mask = (dist2 <= r_outer ** 2) & (dist2 >= r_inner ** 2)
    before[ring_mask] = [198, 165, 87]

    after = before.copy()
    shadow = np.zeros((size, size), dtype=np.float32)
    sy = cy + int(r_outer * 0.9)
    cv2.ellipse(shadow, (cx, sy), (int(r_outer * 0.9), int(r_outer * 0.35)), 0, 0, 360, 1.0, -1)
    shadow = cv2.GaussianBlur(shadow, (31, 31), 0) * 0.35
    for c in range(3):
        after[..., c] = (after[..., c].astype(np.float32) * (1 - shadow)).astype(np.uint8)

    return Image.fromarray(before), Image.fromarray(after)


def build_placeholder_dataset(n_pairs: int = 20, out_dir: str = None):
    """Folder structure per the brief's dataset spec: a 'reference' (control)
    folder of flat hard-pastes and a 'target' folder of retouched versions
    with matching captions -- this is what ai-toolkit's control_path/
    folder_path fields point at.
    """
    out_dir = out_dir or os.path.join(DATASET_DIR, "harmonisation_lora_placeholder")
    ref_dir = os.path.join(out_dir, "reference")
    tgt_dir = os.path.join(out_dir, "target")
    os.makedirs(ref_dir, exist_ok=True)
    os.makedirs(tgt_dir, exist_ok=True)

    for i in range(n_pairs):
        before, after = generate_placeholder_training_pair(seed=i)
        before.save(os.path.join(ref_dir, f"pair_{i:03d}.jpg"), quality=95)
        after.save(os.path.join(tgt_dir, f"pair_{i:03d}.jpg"), quality=95)
        with open(os.path.join(tgt_dir, f"pair_{i:03d}.txt"), "w") as f:
            f.write(HARMONISE_PROMPT)

    print(f"Built {n_pairs} placeholder pairs -> reference: {ref_dir} | target: {tgt_dir}")
    return ref_dir, tgt_dir


print("Placeholder training-data functions ready.")

In [ ]:
import yaml


def generate_lora_training_config(backbone: str, ref_dir: str, tgt_dir: str, output_dir: str,
                                   name: str = None, steps: int = 1500) -> dict:
    """Programmatic ai-toolkit config. Structure verified against ai-toolkit's
    own train_lora_flux_kontext_24gb.yaml (the closest confirmed 'edit LoRA
    with control_path' example). Per-backbone `arch` strings: qwen_image_edit
    family has official ai-toolkit examples (see flag #4); flux_fill's is my
    best guess, NOT independently verified (see flag #3) -- check ai-toolkit's
    current config/examples/ before a real run.
    """
    assert backbone in ("flux_fill", "qwen_edit"), backbone
    name = name or f"skyai_harmonise_{backbone}_v1"
    sample_prompt = f"{HARMONISE_PROMPT} --ctrl_img {ref_dir}/pair_000.jpg"

    if backbone == "flux_fill":
        model_block = {
            "name_or_path": "black-forest-labs/FLUX.1-Fill-dev",
            "arch": "flux_fill",  # ⚠ best-guess, unverified -- flag #3
            "quantize": True,
        }
    else:
        model_block = {
            "name_or_path": "Qwen/Qwen-Image-Edit-2511",
            "arch": "qwen_image_edit_plus",  # matches ai-toolkit's Qwen Edit examples -- flag #4
            "quantize": True,
        }

    return {
        "job": "extension",
        "config": {
            "name": name,
            "process": [{
                "type": "sd_trainer",
                "training_folder": output_dir,  # Drive-persisted -- survives disconnects, flag #7
                "device": "cuda:0",
                "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
                "save": {
                    "dtype": "float16",
                    "save_every": 250,           # brief's checkpoint cadence
                    "max_step_saves_to_keep": 8,
                    "push_to_hub": False,
                },
                "datasets": [{
                    "folder_path": tgt_dir,       # retouched targets + captions
                    "control_path": ref_dir,      # hard-pasted references (no captions needed)
                    "caption_ext": "txt",
                    "caption_dropout_rate": 0.05,
                    "shuffle_tokens": False,
                    "cache_latents_to_disk": True,
                    "resolution": [512, 768],
                }],
                "train": {
                    "batch_size": 1,
                    "steps": steps,
                    "gradient_accumulation_steps": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",     # VRAM headroom -- flag #5
                    "lr": 1e-4,
                    "dtype": "bf16",
                },
                "model": model_block,
                "sample": {
                    "sampler": "flowmatch",
                    "sample_every": 250,
                    "width": 768, "height": 768,
                    "prompts": [sample_prompt],
                    "neg": "",
                    "seed": 42,
                    "walk_seed": True,
                    "guidance_scale": 4,
                    "sample_steps": 20,
                },
            }],
        },
        "meta": {"name": "[name]", "version": "1.0"},
    }


def write_training_config(backbone: str, ref_dir: str, tgt_dir: str, output_dir: str, steps: int = 1500) -> str:
    os.makedirs(output_dir, exist_ok=True)
    config = generate_lora_training_config(backbone, ref_dir, tgt_dir, output_dir, steps=steps)
    config_path = os.path.join(output_dir, f"{config['config']['name']}.yaml")
    with open(config_path, "w") as f:
        yaml.dump(config, f, sort_keys=False)
    return config_path


print("LoRA config generator ready.")

In [ ]:
import subprocess

AI_TOOLKIT_DIR = os.path.join(PROJECT_DIR, "ai-toolkit")  # Drive-persisted clone


def ensure_ai_toolkit() -> str:
    """ai-toolkit is a repo you run, not a library you import -- see flag #6."""
    if not os.path.exists(AI_TOOLKIT_DIR):
        print("Cloning ostris/ai-toolkit ...")
        subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", AI_TOOLKIT_DIR], check=True)
        subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=AI_TOOLKIT_DIR, check=True)
        subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], cwd=AI_TOOLKIT_DIR, check=True)
    else:
        print(f"ai-toolkit already present at {AI_TOOLKIT_DIR}")
    return AI_TOOLKIT_DIR


def run_lora_training(backbone: str, steps: int = 1500, n_placeholder_pairs: int = 20):
    """Writes the config, then shells out to ai-toolkit's run.py.

    Safe to re-run after a Colab disconnect: pointed at the same Drive-
    persisted training_folder/name, ai-toolkit is designed to resume from the
    latest saved checkpoint rather than starting over -- see flag #7 on why
    that specific behaviour wasn't independently verified this session.

    NOTE: not called automatically anywhere in this notebook -- even a smoke
    test downloads the full base model weights through ai-toolkit's own
    mechanism, which is slow and heavy as a default side effect of just
    reading through the notebook. Call it yourself when you're ready:
        run_lora_training("flux_fill", steps=10)   # smoke test
        run_lora_training("qwen_edit", steps=1500) # real run
    """
    ai_toolkit_dir = ensure_ai_toolkit()
    ref_dir, tgt_dir = build_placeholder_dataset(n_pairs=n_placeholder_pairs)
    output_dir = os.path.join(CHECKPOINT_DIR, "lora_training", backbone)

    config_path = write_training_config(backbone, ref_dir, tgt_dir, output_dir, steps=steps)
    print(f"Config written: {config_path}")
    print(
        "Visual quality typically peaks around step 750-1500, not necessarily "
        "the final step -- check the sample_every=250 previews in the output "
        "folder rather than assuming the last checkpoint is the best one."
    )

    cmd = ["python", "run.py", config_path]
    print(f"Launching: {' '.join(cmd)}  (cwd={ai_toolkit_dir})")
    subprocess.run(cmd, cwd=ai_toolkit_dir, check=True)
    return output_dir


print("ai-toolkit runner ready. Not auto-executed -- see this cell's docstring for how to launch it.")

### 11 · Load a trained LoRA here, once one exists

In [ ]:
# Point these at a checkpoint produced by run_lora_training() (see the
# output_dir it prints/returns) once you have one. Left as None for now --
# both harmonise_*() functions run on the base models until you set these.
TRAINED_LORA_PATH_FLUX = None
# e.g. f"{CHECKPOINT_DIR}/lora_training/flux_fill/skyai_harmonise_flux_fill_v1/skyai_harmonise_flux_fill_v1.safetensors"
TRAINED_LORA_PATH_QWEN = None
# e.g. f"{CHECKPOINT_DIR}/lora_training/qwen_edit/skyai_harmonise_qwen_edit_v1/skyai_harmonise_qwen_edit_v1.safetensors"

if TRAINED_LORA_PATH_FLUX and os.path.exists(TRAINED_LORA_PATH_FLUX):
    flux_fill_pipe.load_lora_weights(TRAINED_LORA_PATH_FLUX)
    print(f"Loaded trained LoRA into FLUX.1 Fill: {TRAINED_LORA_PATH_FLUX}")
else:
    print("No trained FLUX LoRA loaded -- harmonise_flux() is running the base model.")

if TRAINED_LORA_PATH_QWEN and os.path.exists(TRAINED_LORA_PATH_QWEN):
    qwen_edit_pipe.load_lora_weights(TRAINED_LORA_PATH_QWEN)
    print(f"Loaded trained LoRA into Qwen-Image-Edit: {TRAINED_LORA_PATH_QWEN}")
else:
    print("No trained Qwen LoRA loaded -- harmonise_qwen() is running the base model.")

### 12 · Full pipeline demo: both backbones, all four stages

Same input, both harmonisation backbones, with a fidelity check on each: does the piece itself actually survive Stage 4 untouched? That's the brief's non-negotiable constraint, tested at the final output rather than assumed.

In [ ]:
def compare_jewellery_region(before_rgb: np.ndarray, after_rgb: np.ndarray, portrait_alpha: np.ndarray) -> dict:
    """Compares the jewellery region between two full-composite images (e.g.
    hard-paste vs. harmonised) -- the real end-to-end test of the brief's
    non-negotiable constraint, checked at the final output rather than
    assumed from the mask design alone. Should score near-perfect for both
    backbones: FLUX via its architectural mask guarantee, Qwen via the
    explicit pixel restore in harmonise_qwen().
    """
    m = portrait_alpha > 0.5
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    before_crop = before_rgb[y0:y1, x0:x1]
    after_crop = np.asarray(after_rgb)[y0:y1, x0:x1]
    crop_mask = m[y0:y1, x0:x1]

    ssim_val = ssim(before_crop, after_crop, channel_axis=2, data_range=255)
    lab1 = rgb2lab(before_crop.astype(np.float64) / 255.0)
    lab2 = rgb2lab(after_crop.astype(np.float64) / 255.0)
    de_map = deltaE_ciede2000(lab1, lab2)
    delta_e_mean = float(de_map[crop_mask].mean())
    lpips_val = compute_lpips(before_crop, after_crop)

    return {"ssim": float(ssim_val), "delta_e": delta_e_mean, "lpips": lpips_val}


print("compare_jewellery_region() ready.")

In [ ]:
if ring_anchor is not None:
    harmonised_flux = harmonise_flux(composite, warped_alpha)
    harmonised_qwen = harmonise_qwen(composite, warped_alpha)

    metrics_flux = compare_jewellery_region(composite, np.array(harmonised_flux), warped_alpha)
    metrics_qwen = compare_jewellery_region(composite, np.array(harmonised_qwen), warped_alpha)

    print_fidelity_report(metrics_flux, label="(Stage 4 -- FLUX.1 Fill, jewellery-region preservation)")
    print()
    print_fidelity_report(metrics_qwen, label="(Stage 4 -- Qwen-Image-Edit, jewellery-region preservation)")

    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    axes[0].imshow(asset_alpha, cmap="gray")
    axes[0].set_title("Stage 1: mask")
    axes[1].imshow(hand_image)
    for label, (x, y) in ring_anchor["debug_points"]:
        axes[1].plot(x, y, "o", markersize=4)
    axes[1].plot(*ring_anchor["anchor_xy"], "r+", markersize=14, markeredgewidth=2)
    axes[1].set_title("Stage 2: anchors")
    axes[2].imshow(composite)
    axes[2].set_title("Stage 3: hard-paste")
    axes[3].imshow(harmonised_flux)
    axes[3].set_title(f"Stage 4: FLUX.1 Fill\nSSIM={metrics_flux['ssim']:.3f} dE={metrics_flux['delta_e']:.2f}")
    axes[4].imshow(harmonised_qwen)
    axes[4].set_title(f"Stage 4: Qwen-Image-Edit\nSSIM={metrics_qwen['ssim']:.3f} dE={metrics_qwen['delta_e']:.2f}")
    for a in axes:
        a.axis("off")
    plt.tight_layout()
    plt.show()

    print(
        "\nHigh SSIM / low dE here confirms the piece survived harmonisation "
        "unaltered (the brief's non-negotiable constraint) -- it does NOT by "
        "itself say which backbone's shadow/lighting looks better. That's a "
        "visual call for you to make from the images above, and ultimately "
        "flag #1's license reality (FLUX dev = non-commercial) may decide it "
        "regardless of which looks better."
    )
else:
    print("No hand detected earlier in Step 2 -- re-run from the hand-image cell with a real photo first.")

## Step 3 status — and the project as a whole

**Built:** FLUX.1 Fill [dev] + `harmonise_flux()` (native pixel-mask guarantee) · Qwen-Image-Edit-2511 + `harmonise_qwen()` (prompt instruction + mandatory pixel restore, since this backbone has no native mask) · a shared boundary-mask builder (dilate minus original, verified to never overlap the jewellery) · placeholder training-pair generation · a parameterized ai-toolkit YAML config generator for both backbones · a subprocess-based training runner with Drive-persisted checkpointing · a clearly marked LoRA-loading spot · a full end-to-end demo running both backbones on the same real photo, with a genuine fidelity check on each.

**The two findings that matter most, business-wise, not just technically:**
1. **FLUX.1 Fill [dev] is non-commercial-licensed** — it cannot be SkyAI's production backbone as deployed here, independent of quality. This is a licensing fact, not a tuning problem.
2. **Qwen-Image-Edit-2511 has no native masking** — I adapted around this (prompt + mandatory pixel restore) rather than pretending "the same masking logic" the brief asked for was literally possible across both backbones. The adaptation is tested and works, but it's a different mechanism, worth understanding before you rely on it.

**What I verified vs. what I'm flagging as unverified:** FLUX.1 Fill's API, license terms, and guidance-scale convention are confirmed against the model's own documentation. Qwen-Image-Edit-2511's pipeline class, lack of mask support, and license are likewise confirmed. The boundary-mask math is tested locally with real geometry (zero overlap with the jewellery, confirmed numerically). The LoRA training YAML structure is verified against ai-toolkit's own Kontext example for the parts common to any edit-style LoRA — but the `flux_fill` arch string is my best guess (unconfirmed), and the Qwen path has a known, possibly-unresolved GitHub issue about control-image handling on this exact model family. Neither harmonisation model nor the training run were actually executed in my build environment — no GPU, and these are multi-GB gated/large downloads. Budget real time for first-run debugging on both, and check the two flagged uncertainties against ai-toolkit's current repo before trusting the training section past a smoke test.

**Across all three steps, if you're deciding what to fix first:** the license issue in flag #1 above is the one thing that changes the shape of the project, not just its quality — worth resolving before investing heavily in FLUX-side tuning. Everything else flagged is a real but bounded accuracy/robustness issue (anthropometric averages, ear/nose-ring landmark approximations, resampling loss on small pieces) that real usage data will tell you whether to fix.